# Phase 10: our own lightning LoRAs, 8 steps and 4 steps

**Base:** version 1, the unhealed 30-block student (B + block-5 attention bridge step 200 + block-20 attention
bridge step 600). The teacher samples in 40 steps without classifier-free guidance; the goal is 8 steps (about 5x
faster) and 4 steps (about 10x), keeping spelling. Reference points: `phase9-lightning-baseline.ipynb`.

**Method: trajectory distillation.** The intact teacher's 40-step trajectories are recorded (latent at every step).
The student's K steps use exact subsets of the teacher's 40-step sigma grid (every 5th point for 8 steps, every 10th
for 4). Training starts the student at a teacher latent and teaches one student step to land where the teacher's 5
(or 10) steps land: the target velocity is (x_next - x_now) / (sigma_next - sigma_now) along the teacher path. A
LoRA on the attention of the 30 original blocks learns it; the bridges stay frozen. Every checkpoint is saved and
judged quantitatively (distance of the K-step final latent from the teacher's 40-step one, on 45 held-out prompts)
and qualitatively (the four-seed text grids and the 25 general prompts).

**Steps:** (1) record the teacher trajectories; (2) train the 8-step LoRA; (3) train the 4-step LoRA; (4) choose
and fuse.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
assert (ROOT / 'PLAN.md').exists(), 'Run this notebook from the repository root.'
sys.path.insert(0, str(ROOT / 'tools'))
for stream in (sys.stdout, sys.stderr):      # Modal prints non-ASCII progress symbols
    if hasattr(stream, 'reconfigure'):
        stream.reconfigure(encoding='utf-8')
ALLOW_GPU = False          # set True only to launch a step whose saved result is missing
SOURCES = {}               # worker sources, embedded per step exactly as executed


def load_summary(run):
    path = ROOT / run / 'summary.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None


def reuse_or_launch(run, launch):
    summary = load_summary(run)
    if summary is not None:
        print('Saved result:', run, '| status', summary.get('status'))
        return summary
    if (ROOT / run).parent.joinpath('aborted.json').exists():
        print('Run stopped by request; nothing to reuse.')
        return None
    if not ALLOW_GPU:
        raise RuntimeError(f'No saved result for {run}; set ALLOW_GPU = True to launch it on Modal.')
    launch()
    return load_summary(run)


def pct(value, digits=2):
    return f'{100 * value:.{digits}f}%'


def velocity_row(label, block):
    o = block['overall']['teacher_relative_l2']
    p = {k: block['panels'][k]['teacher_relative_l2']['mean'] for k in block['panels']}
    s = {k: block['stages'][k]['teacher_relative_l2']['mean'] for k in block['stages']}
    return (f"| {label} | {pct(o['mean'])} | {pct(p['gate10'])} | {pct(p['wider50'])} | "
            f"{pct(s['early'], 1)} / {pct(s['middle'], 1)} / {pct(s['late'], 1)} | {pct(o['p90'], 1)} | {pct(o['maximum'], 1)} |")


def velocity_table(rows):
    head = '| Model | Overall | Check (10) | Wider (50) | Early / middle / late | p90 | Worst |\n|---|---|---|---|---|---|---|\n'
    display(Markdown(head + '\n'.join(rows)))


def show_variants(summary):
    rows = [velocity_row(f"{name} (step {v['fit']['selected_step']}, {v['parameters'] / 1e6:.1f}M params)", v)
            for name, v in summary['variants'].items()]
    velocity_table(rows)


def show_images(folder, names):
    for name in names:
        display(Image(filename=str(ROOT / folder / name), width=1000))

## 1. Teacher trajectories

The intact teacher (32 blocks, 40 steps) on the 500 training prompts of the Phase 7 recordings (450 text, 50 general,
same seeds and sizes) and on the 45 held-out image prompts (5 text prompts with 4 seeds, 25 general prompts; checked
not to overlap). Saved per prompt: the 41 latents (start noise to final), the 41 sigmas, and the prompt conditioning.

In [ ]:
# Teacher trajectories: reuses the saved run; launches GPU work only when its result is missing and ALLOW_GPU is True.
SOURCES['teacher_trajectories_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/teacher_trajectories') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['trajectory']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # The intact 32-block teacher, 40 steps as always; a forward pre-hook records the latent and timestep the transformer
        # receives at every step, and the conditioning of the first step; the step callback records the final latent.
        outputs = {v: {} for v in variants}
        state = {}

        def capture(module, args, kwargs):
            state['steps'].append((kwargs['hidden_states'].detach().to('cpu').clone(), kwargs['timestep'].detach().float().flatten()[0].item()))
            if not state['conditioning']:
                for field in ('encoder_hidden_states', 'encoder_hidden_states_mask', 'img_shapes', 'img_mask'):
                    value = kwargs[field]
                    state['conditioning'][field] = value.detach().to('cpu').clone() if torch.is_tensor(value) else value

        def on_step_end(pipeline, index, timestep, tensors):
            if index == request['teacher_steps'] - 1:
                state['final'] = tensors['latents'].detach().to('cpu').clone()
            return {}

        handle = model.register_forward_pre_hook(capture, with_kwargs=True)
        model.eval()
        for row in request['rows']:
            path = directory / (row['source_id'] + '-trajectory.pt')
            state.update(steps=[], conditioning={}, final=None)
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                     true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                     use_kv_cache=True, output_type='latent', callback_on_step_end=on_step_end)
            torch.cuda.synchronize()
            sigmas = pipe.scheduler.sigmas.detach().float().cpu().clone()
            if len(state['steps']) != request['teacher_steps'] or state['final'] is None or len(sigmas) != request['teacher_steps'] + 1:
                raise RuntimeError('Trajectory length differs: ' + row['source_id'])
            # The pipeline passes (sigma * 1000) in BF16, divided by 1000 in BF16: reproduce that rounding exactly.
            expected = (pipe.scheduler.timesteps.detach().cpu().to(state['steps'][0][0].dtype) / 1000).float().tolist()
            timesteps = [t for _, t in state['steps']]
            if max(abs(t - e) for t, e in zip(timesteps, expected)) > 1e-6 or max(abs(t - s) for t, s in zip(timesteps, sigmas.tolist())) > 1e-2:
                raise RuntimeError('Recorded timesteps differ from the scheduler: ' + row['source_id'])
            latents = torch.stack([h[0] for h, _ in state['steps']] + [state['final'][0]])
            temporary = path.with_suffix('.tmp')
            torch.save({'request_key': key, 'row': row, 'sigmas': sigmas, 'timesteps': timesteps, 'latents': latents,
                        'conditioning': state['conditioning']}, temporary)
            temporary.replace(path)
            outputs['trajectory'][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                       'seconds': time.perf_counter() - started}
            report['calls']['trajectory'] += 1
            if report['calls']['trajectory'] % 25 == 0:
                (directory / 'progress.json').write_text(json.dumps(report, indent=2))
                results_volume.commit()
        handle.remove()
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base = json.loads(next((root / 'data/recovered/modal/phase7/candidate_images').glob('*/request.json')).read_bytes())['request']
    text8 = load('data/phase7/block13_bridge_e2e_text8/ff9d259ae589bdf15b98076930c4d16340ef41f4a2c3db897ea3167e95397054/request.json')['request']
    # Held out: the 5 text prompts with 4 seeds and the 25 general prompts used for every image test.
    text_rows = [r for r in base['rows'] if r['subdimension'] in ('text_rendering', 'posters')]
    held_out = list(text_rows) + [{**r, 'source_id': f"{r['source_id']}-s{k}", 'seed': (r['seed'] + 1000003 * k) % 2**32}
                                  for r in text_rows for k in (1, 2, 3)]
    held_out += [r for r in base['rows'] if r['subdimension'] not in ('text_rendering', 'posters')]
    train = [dict(r, split='train') for r in text8['new_prompts']]
    if {r['prompt'] for r in train} & {r['prompt'] for r in held_out}:
        raise RuntimeError('Training prompts overlap the held-out image prompts.')
    request = {k: base[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                    'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase10-teacher-trajectories-v1', worker_source=SOURCES['teacher_trajectories_worker.py'],
        execution_account='new', candidate_only=True, teacher_steps=40,
        rows=[dict(r, split='held_out') for r in held_out] + train,
        scope='Intact teacher, 40 steps: latent at every step for 500 training prompts (450 text) and the 45 held-out image prompts')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/teacher_trajectories' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/teacher_trajectories/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-teacher-trajectories')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'teacher_trajectories_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN = 'data/phase10/teacher_trajectories/' + main()   # writes the request and returns its key; no GPU
LAUNCH_ARGS = []
summary = reuse_or_launch(RUN, main)
if summary and summary.get('status') == 'passed':
    import statistics
    seconds = [r['trajectory']['seconds'] for r in summary['rows']]
    print(len(seconds), 'trajectories; median seconds', round(statistics.median(seconds), 1),
          '; total GB', round(sum(r['trajectory']['bytes'] for r in summary['rows']) / 1e9, 1))

## 2. The 8-step lightning LoRA

LoRA rank 32 on q, k, v and out of the 30 original blocks (bridges frozen), 2000 updates at lr 1e-4. Every 500 updates:
saved, the endpoint distance on the 45 held-out prompts (quantitative), and the same 45 images (qualitative). Step 0 is
version 1 sampled naively in 8 steps (`data/phase10/lightning_lora/protocol.json`).

In [ ]:
# Lightning LoRA, 8 steps: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
STUDENT_STEPS, BRIDGE_DTYPE = 8, 'bfloat16'
SOURCES['lightning_lora_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/lightning_lora') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'step_{s:04d}' for s in range(0, request['fit']['updates'] + 1, request['fit']['checkpoint_every'])]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Version 1 student with a trainable LoRA; trajectory distillation to K steps; every checkpoint evaluated. ----
        import random
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        from torch.nn.attention import SDPBackend, sdpa_kernel
        K = request['student_steps']
        stride = request['teacher_steps'] // K
        keep = list(range(0, request['teacher_steps'] + 1, stride))
        fit, lora = request['fit'], request['lora']
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(),
                                                     keep_dtype=request['bridge_dtype'] == 'float32')
            bridge_slots.append(bridge_spec['layer'])
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if len(trainable) != 30 * 4 * 2:
            raise RuntimeError('Adapter structure differs.')
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = sum(p.numel() for p in trainable.values())
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()

        # Teacher trajectories: only the K+1 points on the student's grid are kept in memory.
        traj_dir = Path(request['trajectories']['remote_directory'])

        def load_trajectory(source_id):
            path = traj_dir / (source_id + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][source_id]:
                raise RuntimeError('Trajectory differs: ' + source_id)
            return torch.load(path, weights_only=True, map_location='cpu')

        train_items = []
        for source_id in request['trajectories']['train_ids']:
            item = load_trajectory(source_id)
            train_items.append({'source_id': source_id, 'latents': item['latents'][keep].contiguous(), 'sigmas': item['sigmas'][keep],
                                'timesteps': [item['timesteps'][i] for i in keep[:-1]], 'conditioning': item['conditioning']})
        held_final, held_grid = {}, {}
        for row in request['rows']:
            item = load_trajectory(row['source_id'])
            held_final[row['source_id']] = item['latents'][-1].float()
            held_grid[row['source_id']] = item['sigmas'][keep]   # the shift depends on the image size
        report['train_items'] = len(train_items)

        def train_loss(item, segment):
            # One student step from the teacher's latent must land where the teacher's `stride` steps land.
            x = item['latents'][segment].unsqueeze(0).to('cuda')
            x_next = item['latents'][segment + 1].unsqueeze(0).to('cuda')
            sigma, sigma_next = item['sigmas'][segment].item(), item['sigmas'][segment + 1].item()
            target = (x_next.float() - x.float()) / (sigma_next - sigma)
            # Exactly the conditioning the pipeline passed at step 0 (a mask may be None when every token is valid).
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            prediction = model(hidden_states=x, timestep=torch.tensor([item['timesteps'][segment]], device='cuda', dtype=x.dtype),
                               encoder_hidden_states=cond['encoder_hidden_states'],
                               encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                               img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                               kv_cache=QwenImage21KVCache(len(model.transformer_blocks)), kv_cache_mode='extract',
                               return_dict=False)[0][:, -x.shape[1]:].float()
            loss = (prediction - target).square().mean() / target.square().mean().clamp_min(1e-12)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        class SubsetScheduler(FlowMatchEulerDiscreteScheduler):
            # The teacher's 40-step sigma grid (same dynamic shift and terminal stretch), keeping every `stride`-th point.
            stride = 1

            def set_timesteps(self, *args, **kwargs):
                super().set_timesteps(*args, **kwargs)
                self.sigmas = self.sigmas[::self.stride].clone()
                self.timesteps = (self.sigmas[:-1] * self.config.num_train_timesteps).to(self.sigmas.device)
                self.num_inference_steps = len(self.timesteps)

        # The pipeline inspects set_timesteps for a `sigmas` parameter: expose the parent's exact signature.
        import inspect as signature_tools
        SubsetScheduler.set_timesteps.__signature__ = signature_tools.signature(FlowMatchEulerDiscreteScheduler.set_timesteps)
        default_scheduler = pipe.scheduler
        student_scheduler = SubsetScheduler.from_config(default_scheduler.config)
        student_scheduler.stride = stride
        report['checkpoints'] = []
        report['losses'] = []
        started = time.perf_counter()

        def evaluate(step):
            # Quantitative: K-step final latent against the teacher's 40-step final latent (same noise), 45 held-out prompts.
            # Qualitative: the same 45 images (four-seed text grids and 25 general prompts).
            name = f'step_{step:04d}'
            pipe.scheduler = student_scheduler
            errors = {}
            for row in request['rows']:
                final = {}

                def on_step_end(pipeline, index, timestep, tensors):
                    if index == K - 1:
                        final['latents'] = tensors['latents'].detach().float().cpu()
                    return {}

                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil', callback_on_step_end=on_step_end).images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - t0
                if (pipe.scheduler.sigmas.detach().float().cpu() - held_grid[row['source_id']]).abs().max() > 1e-6:
                    raise RuntimeError('Student sigma grid differs from the teacher subset.')
                reference = held_final[row['source_id']]
                error = ((final['latents'][0] - reference).norm() / reference.norm()).item()
                errors[row['source_id']] = error
                path = directory / (row['source_id'] + '-' + name + '.png')
                image.save(path, format='PNG')
                outputs[name][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                   'seconds': seconds, 'endpoint_relative_l2': error}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            text_ids = {r['source_id'] for r in request['rows'] if r['subdimension'] in ('text_rendering', 'posters')}
            mean = lambda values: sum(values) / len(values)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         endpoint_mean=mean(list(errors.values())),
                         endpoint_text=mean([e for s, e in errors.items() if s in text_ids]),
                         endpoint_general=mean([e for s, e in errors.items() if s not in text_ids]))
            if step > 0:
                path = directory / f'lora_{K}steps_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'student_steps': K, 'lora': lora, 'bridge_dtype': request['bridge_dtype'],
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            print(f"{K}-step LoRA, step {step}: endpoint {entry['endpoint_mean']:.4f} (text {entry['endpoint_text']:.4f}, "
                  f"general {entry['endpoint_general']:.4f}) ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        # Smoke test of the training path before any evaluation: one forward and backward, gradients discarded, no update.
        with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
            train_loss(train_items[0], 0).backward()
        if not all(p.grad is not None for p in trainable.values()):
            raise RuntimeError('Some adapters received no gradient.')
        for parameter in trainable.values():
            parameter.grad = None
        optimizer = torch.optim.AdamW(list(trainable.values()), lr=fit['learning_rate'], weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        running = []
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                model.eval()
                evaluate(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    loss = train_loss(rng.choice(train_items), rng.randrange(K)) / fit['accumulation']
                    loss.backward()
                    running.append(loss.item() * fit['accumulation'])
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
            if len(running) >= 50:
                report['losses'].append({'step': step + 1, 'mean_loss': sum(running) / len(running)})
                running = []
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_train():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    traj = load(RUN + '/summary.json')
    traj_request = load(RUN + '/request.json')['request']
    healing = load(next((root / 'data/phase8/healing').glob('*/request.json')).relative_to(root))['request']
    shas = {r['source_id']: r['trajectory']['sha256'] for r in traj['rows']}
    request = {k: traj_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                            'canonical_tables_key', 'core_source', 'wrapper_source', 'teacher_steps')}
    request.update(
        version='phase10-lightning-lora-v1', worker_source=SOURCES['lightning_lora_worker.py'], execution_account='new', candidate_only=True,
        student_steps=STUDENT_STEPS, bridge_dtype=BRIDGE_DTYPE,
        conditioned_bridge_source=healing['conditioned_bridge_source'], components_source=healing['components_source'],
        bridges=healing['bridges'],
        trajectories=dict(remote_directory=f"/phase3/phase10/teacher_trajectories/{traj['request_key']}",
                          train_ids=[r['source_id'] for r in traj_request['rows'] if r['split'] == 'train'], sha256=shas),
        rows=[r for r in traj_request['rows'] if r['split'] == 'held_out'],
        lora=dict(rank=32, alpha=32, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261019, sampling_seed=20261020 + STUDENT_STEPS, learning_rate=1e-4, warmup=50, gradient_clip=1.0,
                 updates=2000, checkpoint_every=500, accumulation=1),
        selection='No automatic selection: every checkpoint saved, endpoint distance on 45 held-out prompts and their images; spelling decides',
        scope=f'{STUDENT_STEPS}-step lightning LoRA (rank 32, attention of the 30 blocks) on version 1, trajectory distillation from the teacher')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/lightning_lora' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/lightning_lora/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App(f'nihonga-phase10-lightning-{STUDENT_STEPS}')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_lora_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'trainable_parameters')}), flush=True)
    return request_key


def lightning_table(report):
    head = '| Checkpoint | Endpoint distance (all 45) | Text (20) | General (25) |\n|---|---|---|---|\n'
    display(Markdown(head + chr(10).join(f"| step {c['step']} | {c['endpoint_mean']:.4f} | {c['endpoint_text']:.4f} | {c['endpoint_general']:.4f} |"
                                         for c in report['checkpoints'])))

LAUNCH_ARGS = ['--dry-run']
RUN_L8 = 'data/phase10/lightning_lora/' + main_train()
LAUNCH_ARGS = []
summary_l8 = reuse_or_launch(RUN_L8, main_train)
if summary_l8 and summary_l8.get('status') == 'passed':
    lightning_table(summary_l8)

## 3. The 4-step lightning LoRA

The same, with one student step for every 10 teacher steps.

In [ ]:
# Lightning LoRA, 4 steps: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
STUDENT_STEPS, BRIDGE_DTYPE = 4, 'bfloat16'
SOURCES['lightning_lora_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/lightning_lora') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'step_{s:04d}' for s in range(0, request['fit']['updates'] + 1, request['fit']['checkpoint_every'])]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Version 1 student with a trainable LoRA; trajectory distillation to K steps; every checkpoint evaluated. ----
        import random
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        from torch.nn.attention import SDPBackend, sdpa_kernel
        K = request['student_steps']
        stride = request['teacher_steps'] // K
        keep = list(range(0, request['teacher_steps'] + 1, stride))
        fit, lora = request['fit'], request['lora']
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(),
                                                     keep_dtype=request['bridge_dtype'] == 'float32')
            bridge_slots.append(bridge_spec['layer'])
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if len(trainable) != 30 * 4 * 2:
            raise RuntimeError('Adapter structure differs.')
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = sum(p.numel() for p in trainable.values())
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()

        # Teacher trajectories: only the K+1 points on the student's grid are kept in memory.
        traj_dir = Path(request['trajectories']['remote_directory'])

        def load_trajectory(source_id):
            path = traj_dir / (source_id + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][source_id]:
                raise RuntimeError('Trajectory differs: ' + source_id)
            return torch.load(path, weights_only=True, map_location='cpu')

        train_items = []
        for source_id in request['trajectories']['train_ids']:
            item = load_trajectory(source_id)
            train_items.append({'source_id': source_id, 'latents': item['latents'][keep].contiguous(), 'sigmas': item['sigmas'][keep],
                                'timesteps': [item['timesteps'][i] for i in keep[:-1]], 'conditioning': item['conditioning']})
        held_final, held_grid = {}, {}
        for row in request['rows']:
            item = load_trajectory(row['source_id'])
            held_final[row['source_id']] = item['latents'][-1].float()
            held_grid[row['source_id']] = item['sigmas'][keep]   # the shift depends on the image size
        report['train_items'] = len(train_items)

        def train_loss(item, segment):
            # One student step from the teacher's latent must land where the teacher's `stride` steps land.
            x = item['latents'][segment].unsqueeze(0).to('cuda')
            x_next = item['latents'][segment + 1].unsqueeze(0).to('cuda')
            sigma, sigma_next = item['sigmas'][segment].item(), item['sigmas'][segment + 1].item()
            target = (x_next.float() - x.float()) / (sigma_next - sigma)
            # Exactly the conditioning the pipeline passed at step 0 (a mask may be None when every token is valid).
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            prediction = model(hidden_states=x, timestep=torch.tensor([item['timesteps'][segment]], device='cuda', dtype=x.dtype),
                               encoder_hidden_states=cond['encoder_hidden_states'],
                               encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                               img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                               kv_cache=QwenImage21KVCache(len(model.transformer_blocks)), kv_cache_mode='extract',
                               return_dict=False)[0][:, -x.shape[1]:].float()
            loss = (prediction - target).square().mean() / target.square().mean().clamp_min(1e-12)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        class SubsetScheduler(FlowMatchEulerDiscreteScheduler):
            # The teacher's 40-step sigma grid (same dynamic shift and terminal stretch), keeping every `stride`-th point.
            stride = 1

            def set_timesteps(self, *args, **kwargs):
                super().set_timesteps(*args, **kwargs)
                self.sigmas = self.sigmas[::self.stride].clone()
                self.timesteps = (self.sigmas[:-1] * self.config.num_train_timesteps).to(self.sigmas.device)
                self.num_inference_steps = len(self.timesteps)

        # The pipeline inspects set_timesteps for a `sigmas` parameter: expose the parent's exact signature.
        import inspect as signature_tools
        SubsetScheduler.set_timesteps.__signature__ = signature_tools.signature(FlowMatchEulerDiscreteScheduler.set_timesteps)
        default_scheduler = pipe.scheduler
        student_scheduler = SubsetScheduler.from_config(default_scheduler.config)
        student_scheduler.stride = stride
        report['checkpoints'] = []
        report['losses'] = []
        started = time.perf_counter()

        def evaluate(step):
            # Quantitative: K-step final latent against the teacher's 40-step final latent (same noise), 45 held-out prompts.
            # Qualitative: the same 45 images (four-seed text grids and 25 general prompts).
            name = f'step_{step:04d}'
            pipe.scheduler = student_scheduler
            errors = {}
            for row in request['rows']:
                final = {}

                def on_step_end(pipeline, index, timestep, tensors):
                    if index == K - 1:
                        final['latents'] = tensors['latents'].detach().float().cpu()
                    return {}

                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil', callback_on_step_end=on_step_end).images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - t0
                if (pipe.scheduler.sigmas.detach().float().cpu() - held_grid[row['source_id']]).abs().max() > 1e-6:
                    raise RuntimeError('Student sigma grid differs from the teacher subset.')
                reference = held_final[row['source_id']]
                error = ((final['latents'][0] - reference).norm() / reference.norm()).item()
                errors[row['source_id']] = error
                path = directory / (row['source_id'] + '-' + name + '.png')
                image.save(path, format='PNG')
                outputs[name][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                   'seconds': seconds, 'endpoint_relative_l2': error}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            text_ids = {r['source_id'] for r in request['rows'] if r['subdimension'] in ('text_rendering', 'posters')}
            mean = lambda values: sum(values) / len(values)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         endpoint_mean=mean(list(errors.values())),
                         endpoint_text=mean([e for s, e in errors.items() if s in text_ids]),
                         endpoint_general=mean([e for s, e in errors.items() if s not in text_ids]))
            if step > 0:
                path = directory / f'lora_{K}steps_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'student_steps': K, 'lora': lora, 'bridge_dtype': request['bridge_dtype'],
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            print(f"{K}-step LoRA, step {step}: endpoint {entry['endpoint_mean']:.4f} (text {entry['endpoint_text']:.4f}, "
                  f"general {entry['endpoint_general']:.4f}) ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        # Smoke test of the training path before any evaluation: one forward and backward, gradients discarded, no update.
        with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
            train_loss(train_items[0], 0).backward()
        if not all(p.grad is not None for p in trainable.values()):
            raise RuntimeError('Some adapters received no gradient.')
        for parameter in trainable.values():
            parameter.grad = None
        optimizer = torch.optim.AdamW(list(trainable.values()), lr=fit['learning_rate'], weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        running = []
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                model.eval()
                evaluate(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup'])
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    loss = train_loss(rng.choice(train_items), rng.randrange(K)) / fit['accumulation']
                    loss.backward()
                    running.append(loss.item() * fit['accumulation'])
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
            if len(running) >= 50:
                report['losses'].append({'step': step + 1, 'mean_loss': sum(running) / len(running)})
                running = []
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_train():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    traj = load(RUN + '/summary.json')
    traj_request = load(RUN + '/request.json')['request']
    healing = load(next((root / 'data/phase8/healing').glob('*/request.json')).relative_to(root))['request']
    shas = {r['source_id']: r['trajectory']['sha256'] for r in traj['rows']}
    request = {k: traj_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                            'canonical_tables_key', 'core_source', 'wrapper_source', 'teacher_steps')}
    request.update(
        version='phase10-lightning-lora-v1', worker_source=SOURCES['lightning_lora_worker.py'], execution_account='new', candidate_only=True,
        student_steps=STUDENT_STEPS, bridge_dtype=BRIDGE_DTYPE,
        conditioned_bridge_source=healing['conditioned_bridge_source'], components_source=healing['components_source'],
        bridges=healing['bridges'],
        trajectories=dict(remote_directory=f"/phase3/phase10/teacher_trajectories/{traj['request_key']}",
                          train_ids=[r['source_id'] for r in traj_request['rows'] if r['split'] == 'train'], sha256=shas),
        rows=[r for r in traj_request['rows'] if r['split'] == 'held_out'],
        lora=dict(rank=32, alpha=32, targets=['to_q', 'to_k', 'to_v', 'to_out.0']),
        fit=dict(init_seed=20261019, sampling_seed=20261020 + STUDENT_STEPS, learning_rate=1e-4, warmup=50, gradient_clip=1.0,
                 updates=2000, checkpoint_every=500, accumulation=1),
        selection='No automatic selection: every checkpoint saved, endpoint distance on 45 held-out prompts and their images; spelling decides',
        scope=f'{STUDENT_STEPS}-step lightning LoRA (rank 32, attention of the 30 blocks) on version 1, trajectory distillation from the teacher')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/lightning_lora' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/lightning_lora/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App(f'nihonga-phase10-lightning-{STUDENT_STEPS}')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_lora_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'trainable_parameters')}), flush=True)
    return request_key


def lightning_table(report):
    head = '| Checkpoint | Endpoint distance (all 45) | Text (20) | General (25) |\n|---|---|---|---|\n'
    display(Markdown(head + chr(10).join(f"| step {c['step']} | {c['endpoint_mean']:.4f} | {c['endpoint_text']:.4f} | {c['endpoint_general']:.4f} |"
                                         for c in report['checkpoints'])))

LAUNCH_ARGS = ['--dry-run']
RUN_L4 = 'data/phase10/lightning_lora/' + main_train()
LAUNCH_ARGS = []
summary_l4 = reuse_or_launch(RUN_L4, main_train)
if summary_l4 and summary_l4.get('status') == 'passed':
    lightning_table(summary_l4)

## 4. Where the text is: weighting letters in the loss

Letters cover only a few percent of the latent, so neither the training loss nor the endpoint distance notices a
misspelled sign. Each teacher training image (the decoded final latent of its trajectory) is read by an OCR detector
(EasyOCR, Chinese and English); every word box is mapped onto the latent token grid (one token = 16x16 pixels) and
widened by one token. The continuation in step 5 weights these tokens more in the loss. 16 previews with the marked
tokens are saved for a visual check.

In [ ]:
# Text regions: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
SOURCES['text_regions_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/text_regions') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['regions']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # Decode each recorded teacher final latent exactly as the pipeline does, detect words with OCR, and mark the
        # latent tokens (16x16-pixel patches) that hold text, widened by one token.
        import math
        import easyocr
        outputs = {v: {} for v in variants}
        reader = easyocr.Reader(request['ocr']['languages'], gpu=True, verbose=False)
        traj_dir = Path(request['trajectories']['remote_directory'])
        latents_mean = torch.tensor(pipe.vae.config.latents_mean).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        latents_std = torch.tensor(pipe.vae.config.latents_std).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        masks = {}
        for row in request['rows']:
            path = traj_dir / (row['source_id'] + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][row['source_id']]:
                raise RuntimeError('Trajectory differs: ' + row['source_id'])
            item = torch.load(path, weights_only=True, map_location='cpu')
            height, width = row['height'], row['width']
            with torch.inference_mode():
                latent = item['latents'][-1].unsqueeze(0).to('cuda')
                latent = pipe._unpack_latents(latent, height, width, pipe.vae_scale_factor).to(pipe.vae.dtype)
                latent = latent * latents_std.to(latent.device, latent.dtype) + latents_mean.to(latent.device, latent.dtype)
                decoded = pipe.vae.decode(latent, return_dict=False)[0][:, :, 0]
                image = pipe.image_processor.postprocess(decoded, output_type='pil')[0].convert('RGB')
            import numpy
            detections = reader.readtext(numpy.asarray(image), min_size=request['ocr']['min_size'])
            rows_t, cols_t = height // 16, width // 16
            mask = torch.zeros(rows_t, cols_t, dtype=torch.bool)
            words = []
            for box, text, confidence in detections:
                if confidence < request['ocr']['min_confidence']:
                    continue
                xs, ys = [p[0] for p in box], [p[1] for p in box]
                c0, c1 = max(0, math.floor(min(xs) / 16) - 1), min(cols_t, math.ceil(max(xs) / 16) + 1)
                r0, r1 = max(0, math.floor(min(ys) / 16) - 1), min(rows_t, math.ceil(max(ys) / 16) + 1)
                mask[r0:r1, c0:c1] = True
                words.append({'text': text, 'confidence': float(confidence), 'box': [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))]})
            if mask.numel() != item['latents'].shape[1]:
                raise RuntimeError('Token grid differs from the latent: ' + row['source_id'])
            masks[row['source_id']] = mask.flatten()
            entry = {'words': words, 'text_tokens': int(mask.sum()), 'tokens': mask.numel()}
            if row['source_id'] in request['preview_ids']:
                from PIL import ImageDraw
                draw = ImageDraw.Draw(image)
                for r_index in range(rows_t):
                    for c_index in range(cols_t):
                        if mask[r_index, c_index]:
                            draw.rectangle([c_index * 16, r_index * 16, c_index * 16 + 15, r_index * 16 + 15], outline=(255, 0, 0))
                preview = directory / (row['source_id'] + '-ocr.jpg')
                image.save(preview, quality=85)
                entry['preview'] = {'name': preview.name, 'sha256': digest(preview), 'bytes': preview.stat().st_size}
            outputs['regions'][row['source_id']] = entry
            report['calls']['regions'] += 1
        path = directory / 'text_masks.pt'
        torch.save({'request_key': key, 'masks': masks, 'token_size': 16, 'dilation': 1}, path)
        report['masks'] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size}
        report['artifacts'].append(report['masks'])
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']]['preview'] for v in rendered if 'preview' in outputs[v][row['source_id']])
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_ocr():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    traj = load(OCR_SOURCE + '/summary.json')
    traj_request = load(OCR_SOURCE + '/request.json')['request']
    train = [r for r in traj_request['rows'] if r['split'] == 'train']
    request = {k: traj_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                            'canonical_tables_key', 'core_source', 'wrapper_source')}
    request.update(
        version='phase10-text-regions-v1', worker_source=SOURCES['text_regions_worker.py'], execution_account='new', candidate_only=True,
        rows=train, preview_ids=[r['source_id'] for r in train if r['subdimension'] == 'text_rendering'][:16],
        trajectories=dict(remote_directory=f"/phase3/phase10/{OCR_SOURCE.split('/')[2]}/{traj['request_key']}",
                          sha256={r['source_id']: r['trajectory']['sha256'] for r in traj['rows']}),
        ocr=dict(package='easyocr==1.7.2', languages=['ch_sim', 'en'], min_confidence=0.3, min_size=10),
        scope='Text regions of the 500 teacher training images: OCR word boxes mapped onto the 16-pixel latent token grid, widened by one token')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request['execution_packages'] = list(request['execution_packages']) + [request['ocr']['package']]
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/text_regions' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/text_regions/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-text-regions')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git', 'libgl1', 'libglib2.0-0')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'text_regions_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=28800, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key


def ocr_summary(report):
    import statistics
    text_rows = [r for r in report['rows'] if r['subdimension'] == 'text_rendering']
    share = [r['regions']['text_tokens'] / r['regions']['tokens'] for r in text_rows]
    found = sum(1 for r in text_rows if r['regions']['text_tokens'] > 0)
    print(f"text prompts with detected text: {found} of {len(text_rows)}; median share of text tokens {100 * statistics.median(share):.1f}%")
    general = [r for r in report['rows'] if r['subdimension'] != 'text_rendering']
    print(f"general prompts with detected text: {sum(1 for r in general if r['regions']['text_tokens'] > 0)} of {len(general)}")


def fetch_ocr_previews(run):
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    out = ROOT / run / 'previews'
    out.mkdir(exist_ok=True)
    for row in json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))['rows']:
        preview = row['regions'].get('preview')
        if preview and not (out / preview['name']).exists():
            (out / preview['name']).write_bytes(b''.join(volume.read_file(f"/phase10/text_regions/{key}/{preview['name']}")))
    return out

OCR_SOURCE = RUN
LAUNCH_ARGS = ['--dry-run']
RUN_OCR = 'data/phase10/text_regions/' + main_ocr()
LAUNCH_ARGS = []
summary_ocr = reuse_or_launch(RUN_OCR, main_ocr)
if summary_ocr and summary_ocr.get('status') == 'passed':
    ocr_summary(summary_ocr)
    previews = fetch_ocr_previews(RUN_OCR)
    show_images(str(previews.relative_to(ROOT)), sorted(p.name for p in previews.glob('*.jpg'))[:8])

## 5. Continuing the 8-step LoRA with text-weighted loss

Steps 2-3 showed a plateau: the 8-step LoRA spells 5-6 of 8 headline signs from step 500 on, its loss stopped falling
near step 1000, and the learning rate never decayed. The best checkpoint (step 1500) is continued for 2000 updates
with three changes: text tokens (step 4) weigh 10 times more in the loss, the learning rate decays to zero on a cosine,
and each update averages 2 examples. Step 0 of this run re-evaluates step 1500 (it must reproduce its endpoint
distance, 0.3769). Every 500 updates: saved, endpoint distance and the 45 images.

In [ ]:
# Continuation: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run steps 1, 2 and 4 first).
CONTINUE_FROM = 1500
SOURCES['lightning_continued_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/lightning_continued') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'step_{s:04d}' for s in range(0, request['fit']['updates'] + 1, request['fit']['checkpoint_every'])]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Version 1 student with a trainable LoRA; trajectory distillation to K steps; every checkpoint evaluated. ----
        import math
        import random
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        from torch.nn.attention import SDPBackend, sdpa_kernel
        K = request['student_steps']
        stride = request['teacher_steps'] // K
        keep = list(range(0, request['teacher_steps'] + 1, stride))
        fit, lora = request['fit'], request['lora']
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(),
                                                     keep_dtype=request['bridge_dtype'] == 'float32')
            bridge_slots.append(bridge_spec['layer'])
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if len(trainable) != 30 * 4 * 2:
            raise RuntimeError('Adapter structure differs.')
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = sum(p.numel() for p in trainable.values())
        init = request['init_lora']
        init_path = Path(init['remote_path'])
        if digest(init_path) != init['sha256']:
            raise RuntimeError('Initial LoRA differs.')
        init_state = torch.load(init_path, weights_only=True, map_location='cpu')
        if init_state['step'] != init['step'] or init_state['student_steps'] != request['student_steps'] or sorted(init_state['state_dict']) != sorted(trainable):
            raise RuntimeError('Initial LoRA provenance differs.')
        with torch.no_grad():
            for name, parameter in trainable.items():
                parameter.copy_(init_state['state_dict'][name])
        # Text tokens found by OCR in the teacher's training images weigh more in the loss.
        masks_spec = request['text_masks']
        if digest(Path(masks_spec['remote_path'])) != masks_spec['sha256']:
            raise RuntimeError('Text masks differ.')
        text_masks = torch.load(Path(masks_spec['remote_path']), weights_only=True, map_location='cpu')['masks']
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()

        # Teacher trajectories: only the K+1 points on the student's grid are kept in memory.
        traj_dir = Path(request['trajectories']['remote_directory'])

        def load_trajectory(source_id):
            path = traj_dir / (source_id + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][source_id]:
                raise RuntimeError('Trajectory differs: ' + source_id)
            return torch.load(path, weights_only=True, map_location='cpu')

        train_items = []
        for source_id in request['trajectories']['train_ids']:
            item = load_trajectory(source_id)
            weight = 1.0 + (request['fit']['text_weight'] - 1.0) * text_masks[source_id].float()
            train_items.append({'source_id': source_id, 'weight': weight, 'latents': item['latents'][keep].contiguous(), 'sigmas': item['sigmas'][keep],
                                'timesteps': [item['timesteps'][i] for i in keep[:-1]], 'conditioning': item['conditioning']})
        held_final, held_grid = {}, {}
        for row in request['rows']:
            item = load_trajectory(row['source_id'])
            held_final[row['source_id']] = item['latents'][-1].float()
            held_grid[row['source_id']] = item['sigmas'][keep]   # the shift depends on the image size
        report['train_items'] = len(train_items)

        def train_loss(item, segment):
            # One student step from the teacher's latent must land where the teacher's `stride` steps land.
            x = item['latents'][segment].unsqueeze(0).to('cuda')
            x_next = item['latents'][segment + 1].unsqueeze(0).to('cuda')
            sigma, sigma_next = item['sigmas'][segment].item(), item['sigmas'][segment + 1].item()
            target = (x_next.float() - x.float()) / (sigma_next - sigma)
            # Exactly the conditioning the pipeline passed at step 0 (a mask may be None when every token is valid).
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            prediction = model(hidden_states=x, timestep=torch.tensor([item['timesteps'][segment]], device='cuda', dtype=x.dtype),
                               encoder_hidden_states=cond['encoder_hidden_states'],
                               encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                               img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                               kv_cache=QwenImage21KVCache(len(model.transformer_blocks)), kv_cache_mode='extract',
                               return_dict=False)[0][:, -x.shape[1]:].float()
            weight = item['weight'].to('cuda').view(1, -1, 1)
            loss = (weight * (prediction - target).square()).sum() / (weight * target.square()).sum().clamp_min(1e-12)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        class SubsetScheduler(FlowMatchEulerDiscreteScheduler):
            # The teacher's 40-step sigma grid (same dynamic shift and terminal stretch), keeping every `stride`-th point.
            stride = 1

            def set_timesteps(self, *args, **kwargs):
                super().set_timesteps(*args, **kwargs)
                self.sigmas = self.sigmas[::self.stride].clone()
                self.timesteps = (self.sigmas[:-1] * self.config.num_train_timesteps).to(self.sigmas.device)
                self.num_inference_steps = len(self.timesteps)

        # The pipeline inspects set_timesteps for a `sigmas` parameter: expose the parent's exact signature.
        import inspect as signature_tools
        SubsetScheduler.set_timesteps.__signature__ = signature_tools.signature(FlowMatchEulerDiscreteScheduler.set_timesteps)
        default_scheduler = pipe.scheduler
        student_scheduler = SubsetScheduler.from_config(default_scheduler.config)
        student_scheduler.stride = stride
        report['checkpoints'] = []
        report['losses'] = []
        started = time.perf_counter()

        def evaluate(step):
            # Quantitative: K-step final latent against the teacher's 40-step final latent (same noise), 45 held-out prompts.
            # Qualitative: the same 45 images (four-seed text grids and 25 general prompts).
            name = f'step_{step:04d}'
            pipe.scheduler = student_scheduler
            errors = {}
            for row in request['rows']:
                final = {}

                def on_step_end(pipeline, index, timestep, tensors):
                    if index == K - 1:
                        final['latents'] = tensors['latents'].detach().float().cpu()
                    return {}

                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil', callback_on_step_end=on_step_end).images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - t0
                if (pipe.scheduler.sigmas.detach().float().cpu() - held_grid[row['source_id']]).abs().max() > 1e-6:
                    raise RuntimeError('Student sigma grid differs from the teacher subset.')
                reference = held_final[row['source_id']]
                error = ((final['latents'][0] - reference).norm() / reference.norm()).item()
                errors[row['source_id']] = error
                path = directory / (row['source_id'] + '-' + name + '.png')
                image.save(path, format='PNG')
                outputs[name][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                   'seconds': seconds, 'endpoint_relative_l2': error}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            text_ids = {r['source_id'] for r in request['rows'] if r['subdimension'] in ('text_rendering', 'posters')}
            mean = lambda values: sum(values) / len(values)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         endpoint_mean=mean(list(errors.values())),
                         endpoint_text=mean([e for s, e in errors.items() if s in text_ids]),
                         endpoint_general=mean([e for s, e in errors.items() if s not in text_ids]))
            if step > 0:
                path = directory / f'lora_{K}steps_continued_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'continued_from': request['init_lora'], 'student_steps': K, 'lora': lora, 'bridge_dtype': request['bridge_dtype'],
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            print(f"{K}-step LoRA (continued), step {step}: endpoint {entry['endpoint_mean']:.4f} (text {entry['endpoint_text']:.4f}, "
                  f"general {entry['endpoint_general']:.4f}) ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        # Smoke test of the training path before any evaluation: one forward and backward, gradients discarded, no update.
        with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
            train_loss(train_items[0], 0).backward()
        if not all(p.grad is not None for p in trainable.values()):
            raise RuntimeError('Some adapters received no gradient.')
        for parameter in trainable.values():
            parameter.grad = None
        optimizer = torch.optim.AdamW(list(trainable.values()), lr=fit['learning_rate'], weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        running = []
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                model.eval()
                evaluate(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                # Short warmup, then cosine decay to zero over the run.
                group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup']) * 0.5 * (1 + math.cos(math.pi * step / fit['updates']))
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    loss = train_loss(rng.choice(train_items), rng.randrange(K)) / fit['accumulation']
                    loss.backward()
                    running.append(loss.item() * fit['accumulation'])
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
            if len(running) >= 50:
                report['losses'].append({'step': step + 1, 'mean_loss': sum(running) / len(running)})
                running = []
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_continue():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base_request = load(RUN_L8 + '/request.json')['request']
    base_summary = load(RUN_L8 + '/summary.json')
    ocr = load(RUN_OCR + '/summary.json')
    start = next(c for c in base_summary['checkpoints'] if c['step'] == CONTINUE_FROM)
    request = {k: v for k, v in base_request.items() if k not in ('worker_source', 'version', 'fit', 'scope', 'selection')}
    request.update(
        version='phase10-lightning-continued-v1', worker_source=SOURCES['lightning_continued_worker.py'],
        init_lora=dict(step=CONTINUE_FROM, remote_path=f"/phase3/phase10/lightning_lora/{base_summary['request_key']}/{start['checkpoint']['name']}",
                       sha256=start['checkpoint']['sha256']),
        text_masks=dict(remote_path=f"/phase3/phase10/text_regions/{ocr['request_key']}/{ocr['masks']['name']}", sha256=ocr['masks']['sha256']),
        fit=dict(init_seed=20261019, sampling_seed=20261028, learning_rate=1e-4, warmup=20, gradient_clip=1.0,
                 updates=2000, checkpoint_every=500, accumulation=2, text_weight=10.0),
        selection='No automatic selection: every checkpoint saved, endpoint distance on 45 held-out prompts and their images; spelling decides',
        scope=f'8-step LoRA continued from step {CONTINUE_FROM}: text tokens weighted 10x, cosine-decaying learning rate, 2 examples per update')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/lightning_continued' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/lightning_continued/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-lightning-continued')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_continued_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=36000, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'trainable_parameters')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN_CONT = 'data/phase10/lightning_continued/' + main_continue()
LAUNCH_ARGS = []
summary_cont = reuse_or_launch(RUN_CONT, main_continue)
if summary_cont and summary_cont.get('status') == 'passed':
    lightning_table(summary_cont)

## 5b. Comparison sheets

Local, no GPU. Text prompts with seeds 0-3 and the 25 general prompts with seed 0: teacher (40 steps), version 1 at 40
steps (the ceiling for a LoRA on version 1), the LightX2V 4-step LoRA on the teacher, our 8-step LoRA at step 1500, and
every checkpoint of the continuation.

In [ ]:
# Lightning comparison sheets: local only (run steps 2 and 5 first; Phase 9 images must be in .archive).
def fetch_run_images(run, remote_root):
    """Download a run's images into .archive, checking every hash."""
    import hashlib
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key = run.rstrip('/').split('/')[-1]
    store = ROOT / '.archive' / remote_root.strip('/').replace('/', '_') / key
    store.mkdir(parents=True, exist_ok=True)
    summary = json.loads((ROOT / run / 'summary.json').read_text(encoding='utf-8'))
    for row in summary['rows']:
        for value in row.values():
            if isinstance(value, dict) and str(value.get('name', '')).endswith('.png'):
                path = store / value['name']
                if not path.exists():
                    path.write_bytes(b''.join(volume.read_file(f"{remote_root}/{key}/{value['name']}")))
                if hashlib.sha256(path.read_bytes()).hexdigest() != value['sha256']:
                    raise RuntimeError('Image hash mismatch: ' + value['name'])
    return store


def build_lightning_sheets(run_l8, run_cont, out):
    """Text prompts (seeds 0-3) and general prompts (seed 0): teacher 40, version 1 at 40 (BF16 bridges), LightX2V 4,
    our 8-step LoRA at step 1500, and every checkpoint of the continuation."""
    import textwrap
    from PIL import Image as PILImage, ImageDraw, ImageFont
    l8 = fetch_run_images(run_l8, '/phase10/lightning_lora')
    cont = fetch_run_images(run_cont, '/phase10/lightning_continued')
    cont_steps = [c['step'] for c in json.loads((ROOT / run_cont / 'summary.json').read_text(encoding='utf-8'))['checkpoints'] if c['step'] > 0]
    first = json.loads(next((ROOT / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    first_store = ROOT / '.archive/bridge_images' / first['request_key']
    reference = ROOT / '.archive/multiseed_text/af4a34ba1a0c739121706991b89e79ec92bff3debaf1b1ddbf5b6188c182f514'
    v1 = next((ROOT / '.archive/phase9_bf16_bridges').iterdir())
    lightx = next((ROOT / '.archive/phase9_lightning_baseline').iterdir())
    face = next((f for f in ('C:/Windows/Fonts/msyh.ttc', 'C:/Windows/Fonts/arial.ttf') if Path(f).exists()))
    font, big = ImageFont.truetype(face, 18), ImageFont.truetype(face, 21)
    out = ROOT / out
    out.mkdir(parents=True, exist_ok=True)
    for index, row in enumerate(first['rows']):
        sid = row['source_id']
        name = lambda k, suffix: f'{sid}-{suffix}.png' if k == 0 else f'{sid}-s{k}-{suffix}.png'
        models = [('Teacher, 40 steps', lambda k: first_store / row['teacher']['name'] if k == 0 else reference / name(k, 'teacher')),
                  ('Version 1, 40 steps', lambda k: v1 / name(k, 'v1_bf16')),
                  ('LightX2V (teacher), 4 steps', lambda k: lightx / name(k, 'teacher_lightx2v4')),
                  ('Ours, 8 steps, step 1500', lambda k: l8 / name(k, 'step_1500'))]
        models += [(f'Ours, 8 steps, continued +{s}', lambda k, s=s: cont / name(k, f'step_{s:04d}')) for s in cont_steps]
        text = row['subdimension'] in ('text_rendering', 'posters')
        seeds = 4 if text else 1
        tile, label = (300, 330) if text else (360, 0)
        if text:
            canvas = PILImage.new('RGB', (label + seeds * tile, 80 + len(models) * tile), 'white')
            draw = ImageDraw.Draw(canvas)
            draw.text((10, 8), textwrap.shorten(row['prompt'], 190), font=font, fill='black')
            for column in range(seeds):
                draw.text((label + column * tile + 10, 48), f'seed {column}', font=big, fill='black')
            for line, (title, locate) in enumerate(models):
                y = 80 + line * tile
                draw.text((10, y + tile // 2 - 12), title, font=big, fill='black')
                for column in range(seeds):
                    with PILImage.open(locate(column)) as image:
                        canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (label + column * tile, y))
        else:
            canvas = PILImage.new('RGB', (len(models) * tile, 100 + tile), 'white')
            draw = ImageDraw.Draw(canvas)
            draw.text((10, 8), textwrap.shorten(f'P{index + 1:02d}  ' + row['prompt'], 200), font=font, fill='black')
            for column, (title, locate) in enumerate(models):
                draw.text((column * tile + 6, 50), textwrap.shorten(title, 34), font=ImageFont.truetype(face, 16), fill='black')
                with PILImage.open(locate(0)) as image:
                    canvas.paste(image.convert('RGB').resize((tile, tile), PILImage.Resampling.LANCZOS), (column * tile, 100))
        canvas.save(out / f'P{index + 1:02d}.jpg', quality=88)
    return out

if summary_cont and summary_cont.get('status') == 'passed':
    sheets = build_lightning_sheets(RUN_L8, RUN_CONT, RUN_CONT + '/sheets')
    show_images(str(sheets.relative_to(ROOT)), sorted(p.name for p in sheets.glob('*.jpg')))

### What the results mean (steps 2-5)

**Distance to the teacher's 40-step result** (relative L2 of the final latent, 45 held-out prompts):

| Run | Step 0 | 500 | 1000 | 1500 | 2000 |
|---|---|---|---|---|---|
| 8 steps (text) | 38.9% | 35.8% | 35.5% | 37.9% | 39.2% |
| 4 steps (text) | 48.0% | 43.5% | 35.0% | 34.8% | 33.8% |
| 8 steps continued from 1500, text-weighted (text) | 37.9% | 34.6% | 33.3% | **31.6%** | 31.9% |

**Spelling** (4 seeds each of the Rusty Anchor headline, the Emma headline, the poster tagline and the handwritten note; 16 items):

| Model | Anchor | Emma | Tagline | Note | Total |
|---|---|---|---|---|---|
| teacher, 40 steps | 4 | 4 | 4 | 4 | 16 |
| version 1, 40 steps | 4 | 4 | 3 | 4 | **15** |
| LightX2V on the teacher, 4 steps | 1 | 4 | 4 | 4 | 13 |
| ours, 8 steps, step 1500 | 3 | 3 | 2 | 3 | 11 |
| ours, continued +1000 | 2 | 4 | 3 | 3 | 12 |
| ours, continued +2000 | 2 | 4 | 3 | 3 | **12** |

(review_continued.json.) The first 8-step run learned the teacher's layouts and most letters by step 500 and then
plateaued (loss flat from step 1000; constant learning rate). The 4-step run stays soft and greyish, well behind
LightX2V on image quality: matching the teacher's path averages at 4 steps, which a distribution-matching method avoids.
The continuation with text weighted 10 times (OCR found text in 370 of 450 text images, median 12% of tokens), a
cosine-decaying learning rate and 2 examples per update lowered the text distance to 31.6-31.9% and made Emma and the
poster tagline more reliable, but the Rusty Anchor still fails in two of four seeds at every checkpoint. General images
are unchanged by the continuation; against version 1 at 40 steps the 8-step LoRA loses some structure (the star, the
four friends, the alarm clock).

**Current best 8-step LoRA: continued +2000** (lowest general distance, 12 of 16 items). It is about 4.5 times faster
than 40 steps and spells better than LightX2V on the Rusty Anchor, but below version 1 at 40 steps. Next options:
learn from version 1's own trajectories (the LoRA then only compresses steps), more text data, or training from the
student's own states.


## 6. Version 1's own trajectories, balanced prompts

The 8-step LoRA trained on the teacher's trajectories breaks structure on about 5 of the 25 general prompts (a star
becomes a blob, four friends a crowd, a cyclist a ghost, the alarm clock disappears): it is asked to reach the
32-block teacher's images, which version 1 cannot draw even in 40 steps, and averages. Here the targets are version 1's
own 40-step trajectories (BF16 bridges, as it ships), so the LoRA only has to compress steps; the ceiling becomes
version 1 at 40 steps. The prompts are rebalanced to 500 text and 500 general: the previous 500 plus 50 text and 450
general prompts from the Phase 1 training split, round-robin over every non-text subdimension, checked not to overlap
the 45 held-out prompts.

In [ ]:
# Version 1 trajectories: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 1 first).
SOURCES['v1_trajectories_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/v1_trajectories') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = ['trajectory']
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # The intact 32-block teacher, 40 steps as always; a forward pre-hook records the latent and timestep the transformer
        # receives at every step, and the conditioning of the first step; the step callback records the final latent.
        outputs = {v: {} for v in variants}
        state = {}

        def capture(module, args, kwargs):
            state['steps'].append((kwargs['hidden_states'].detach().to('cpu').clone(), kwargs['timestep'].detach().float().flatten()[0].item()))
            if not state['conditioning']:
                for field in ('encoder_hidden_states', 'encoder_hidden_states_mask', 'img_shapes', 'img_mask'):
                    value = kwargs[field]
                    state['conditioning'][field] = value.detach().to('cpu').clone() if torch.is_tensor(value) else value

        def on_step_end(pipeline, index, timestep, tensors):
            if index == request['teacher_steps'] - 1:
                state['final'] = tensors['latents'].detach().to('cpu').clone()
            return {}

        # Version 1: blocks 5 and 20 replaced by their attention bridges, in BF16 as it ships.
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=False)
        if any(p.dtype != torch.bfloat16 for spec in request['bridges'] for p in model.transformer_blocks[spec['layer']].parameters()):
            raise RuntimeError('Bridges are not BF16.')
        gc.collect()
        torch.cuda.empty_cache()
        handle = model.register_forward_pre_hook(capture, with_kwargs=True)
        model.eval()
        for row in request['rows']:
            path = directory / (row['source_id'] + '-trajectory.pt')
            state.update(steps=[], conditioning={}, final=None)
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                     true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                     use_kv_cache=True, output_type='latent', callback_on_step_end=on_step_end)
            torch.cuda.synchronize()
            sigmas = pipe.scheduler.sigmas.detach().float().cpu().clone()
            if len(state['steps']) != request['teacher_steps'] or state['final'] is None or len(sigmas) != request['teacher_steps'] + 1:
                raise RuntimeError('Trajectory length differs: ' + row['source_id'])
            # The pipeline passes (sigma * 1000) in BF16, divided by 1000 in BF16: reproduce that rounding exactly.
            expected = (pipe.scheduler.timesteps.detach().cpu().to(state['steps'][0][0].dtype) / 1000).float().tolist()
            timesteps = [t for _, t in state['steps']]
            if max(abs(t - e) for t, e in zip(timesteps, expected)) > 1e-6 or max(abs(t - s) for t, s in zip(timesteps, sigmas.tolist())) > 1e-2:
                raise RuntimeError('Recorded timesteps differ from the scheduler: ' + row['source_id'])
            latents = torch.stack([h[0] for h, _ in state['steps']] + [state['final'][0]])
            temporary = path.with_suffix('.tmp')
            torch.save({'request_key': key, 'row': row, 'sigmas': sigmas, 'timesteps': timesteps, 'latents': latents,
                        'conditioning': state['conditioning']}, temporary)
            temporary.replace(path)
            outputs['trajectory'][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                       'seconds': time.perf_counter() - started}
            report['calls']['trajectory'] += 1
            if report['calls']['trajectory'] % 25 == 0:
                (directory / 'progress.json').write_text(json.dumps(report, indent=2))
                results_volume.commit()
        handle.remove()
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_v1():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    teacher_request = load(RUN + '/request.json')['request']
    healing = load(next((root / 'data/phase8/healing').glob('*/request.json')).relative_to(root))['request']
    held_out = [r for r in teacher_request['rows'] if r['split'] == 'held_out']
    old_train = [r for r in teacher_request['rows'] if r['split'] == 'train']
    sizes = {'1:1': (1024, 1024), '16:9': (1376, 768), '9:16': (768, 1376), '4:3': (1184, 896), '3:4': (896, 1184), '3:2': (1248, 832), '2:3': (832, 1248)}
    pool = [json.loads(line) for line in (root / 'data/phase1/pilot_20k/train.jsonl').read_text(encoding='utf-8').splitlines()]
    used = {r['source_id'] for r in old_train} | {r['source_id'].split('-s')[0] for r in held_out}
    banned = {r['prompt'] for r in held_out}
    fresh = sorted((p for p in pool if p['id'] not in used and p['prompt'] not in banned and p['aspect_ratio'] in sizes), key=lambda p: p['pool_rank'])
    as_row = lambda p: dict(source_id=p['id'], prompt=p['prompt'], seed=p['seed'], dimension=p['dimension'], subdimension=p['subdimension'],
                            width=sizes[p['aspect_ratio']][0], height=sizes[p['aspect_ratio']][1], split='train')
    old_text = [r for r in old_train if r['subdimension'] == 'text_rendering']
    old_general = [r for r in old_train if r['subdimension'] != 'text_rendering']
    new_text = [as_row(p) for p in fresh if p['subdimension'] == 'text_rendering'][:500 - len(old_text)]
    # General prompts: round-robin over every non-text subdimension, in saved pool order.
    by_sub = {}
    for p in fresh:
        if p['subdimension'] not in ('text_rendering', 'posters'):
            by_sub.setdefault(p['subdimension'], []).append(p)
    new_general, depth = [], 0
    while len(old_general) + len(new_general) < 500:
        for sub in sorted(by_sub):
            if depth < len(by_sub[sub]) and len(old_general) + len(new_general) < 500:
                new_general.append(as_row(by_sub[sub][depth]))
        depth += 1
    train = old_text + new_text + old_general + new_general
    if len(train) != 1000 or len({r['source_id'] for r in train}) != 1000:
        raise RuntimeError('Training prompt set differs.')
    request = {k: teacher_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                               'canonical_tables_key', 'core_source', 'wrapper_source', 'teacher_steps')}
    request.update(
        version='phase10-v1-trajectories-v1', worker_source=SOURCES['v1_trajectories_worker.py'], execution_account='new', candidate_only=True,
        conditioned_bridge_source=healing['conditioned_bridge_source'], bridges=healing['bridges'],
        rows=held_out + train,
        scope='Version 1 (BF16 bridges), 40 steps: latent at every step for 500 text and 500 general training prompts and the 45 held-out prompts')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/v1_trajectories' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/v1_trajectories/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-v1-trajectories')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'v1_trajectories_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=36000, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN_V1 = 'data/phase10/v1_trajectories/' + main_v1()
LAUNCH_ARGS = []
summary_v1 = reuse_or_launch(RUN_V1, main_v1)

## 7. Text regions in version 1's images

The same OCR step as step 4, on version 1's final images of the 1000 training prompts.

In [ ]:
# Text regions for version 1: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run steps 4 and 6 first).
OCR_SOURCE = RUN_V1
LAUNCH_ARGS = ['--dry-run']
RUN_OCR_V1 = 'data/phase10/text_regions/' + main_ocr()
LAUNCH_ARGS = []
summary_ocr_v1 = reuse_or_launch(RUN_OCR_V1, main_ocr)
if summary_ocr_v1 and summary_ocr_v1.get('status') == 'passed':
    ocr_summary(summary_ocr_v1)

## 8. A fresh 8-step LoRA on version 1's trajectories

From zero (not from the earlier LoRA, so the effect of the new targets is measured cleanly), with the settings of the
best run: text tokens weighted 10 times, cosine-decaying learning rate, 2 examples per update; 3000 updates, a
checkpoint every 500. Each checkpoint reports the distance to version 1's 40-step result (its target) and to the
teacher's, and renders the 45 held-out images.

In [ ]:
# Lightning LoRA on version 1 targets: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run steps 2, 6 and 7 first).
SOURCES['lightning_v1_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/lightning_v1') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'step_{s:04d}' for s in range(0, request['fit']['updates'] + 1, request['fit']['checkpoint_every'])]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Version 1 student with a trainable LoRA; trajectory distillation to K steps; every checkpoint evaluated. ----
        import math
        import random
        from diffusers import FlowMatchEulerDiscreteScheduler
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        from torch.nn.attention import SDPBackend, sdpa_kernel
        K = request['student_steps']
        stride = request['teacher_steps'] // K
        keep = list(range(0, request['teacher_steps'] + 1, stride))
        fit, lora = request['fit'], request['lora']
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(),
                                                     keep_dtype=request['bridge_dtype'] == 'float32')
            bridge_slots.append(bridge_spec['layer'])
        torch.manual_seed(fit['init_seed'])
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), lora['rank'], lora['alpha']))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], lora['rank'], lora['alpha'])
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        model.requires_grad_(False)
        model.gradient_checkpointing = False
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if len(trainable) != 30 * 4 * 2:
            raise RuntimeError('Adapter structure differs.')
        for parameter in trainable.values():
            parameter.requires_grad_(True)
        report['trainable_parameters'] = sum(p.numel() for p in trainable.values())
        init = request.get('init_lora')
        init_path = Path(init['remote_path']) if init else None
        if init:
            if digest(init_path) != init['sha256']:
                raise RuntimeError('Initial LoRA differs.')
            init_state = torch.load(init_path, weights_only=True, map_location='cpu')
            if init_state['step'] != init['step'] or init_state['student_steps'] != request['student_steps'] or sorted(init_state['state_dict']) != sorted(trainable):
                raise RuntimeError('Initial LoRA provenance differs.')
            with torch.no_grad():
                for name, parameter in trainable.items():
                    parameter.copy_(init_state['state_dict'][name])
        # Text tokens found by OCR in the teacher's training images weigh more in the loss.
        masks_spec = request['text_masks']
        if digest(Path(masks_spec['remote_path'])) != masks_spec['sha256']:
            raise RuntimeError('Text masks differ.')
        text_masks = torch.load(Path(masks_spec['remote_path']), weights_only=True, map_location='cpu')['masks']
        gc.collect()
        torch.cuda.empty_cache()
        model.eval()

        # Teacher trajectories: only the K+1 points on the student's grid are kept in memory.
        traj_dir = Path(request['trajectories']['remote_directory'])

        def load_trajectory(source_id):
            path = traj_dir / (source_id + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][source_id]:
                raise RuntimeError('Trajectory differs: ' + source_id)
            return torch.load(path, weights_only=True, map_location='cpu')

        train_items = []
        for source_id in request['trajectories']['train_ids']:
            item = load_trajectory(source_id)
            weight = 1.0 + (request['fit']['text_weight'] - 1.0) * text_masks[source_id].float()
            train_items.append({'source_id': source_id, 'weight': weight, 'latents': item['latents'][keep].contiguous(), 'sigmas': item['sigmas'][keep],
                                'timesteps': [item['timesteps'][i] for i in keep[:-1]], 'conditioning': item['conditioning']})
        held_final, held_grid, teacher_final = {}, {}, {}
        for row in request['rows']:
            item = load_trajectory(row['source_id'])
            held_final[row['source_id']] = item['latents'][-1].float()
            teacher_ref = request['teacher_reference']
            teacher_path = Path(teacher_ref['remote_directory']) / (row['source_id'] + '-trajectory.pt')
            if digest(teacher_path) != teacher_ref['sha256'][row['source_id']]:
                raise RuntimeError('Teacher reference differs: ' + row['source_id'])
            teacher_final[row['source_id']] = torch.load(teacher_path, weights_only=True, map_location='cpu')['latents'][-1].float()
            held_grid[row['source_id']] = item['sigmas'][keep]   # the shift depends on the image size
        report['train_items'] = len(train_items)

        def train_loss(item, segment):
            # One student step from the teacher's latent must land where the teacher's `stride` steps land.
            x = item['latents'][segment].unsqueeze(0).to('cuda')
            x_next = item['latents'][segment + 1].unsqueeze(0).to('cuda')
            sigma, sigma_next = item['sigmas'][segment].item(), item['sigmas'][segment + 1].item()
            target = (x_next.float() - x.float()) / (sigma_next - sigma)
            # Exactly the conditioning the pipeline passed at step 0 (a mask may be None when every token is valid).
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            prediction = model(hidden_states=x, timestep=torch.tensor([item['timesteps'][segment]], device='cuda', dtype=x.dtype),
                               encoder_hidden_states=cond['encoder_hidden_states'],
                               encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                               img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                               kv_cache=QwenImage21KVCache(len(model.transformer_blocks)), kv_cache_mode='extract',
                               return_dict=False)[0][:, -x.shape[1]:].float()
            weight = item['weight'].to('cuda').view(1, -1, 1)
            loss = (weight * (prediction - target).square()).sum() / (weight * target.square()).sum().clamp_min(1e-12)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite training loss.')
            return loss

        class SubsetScheduler(FlowMatchEulerDiscreteScheduler):
            # The teacher's 40-step sigma grid (same dynamic shift and terminal stretch), keeping every `stride`-th point.
            stride = 1

            def set_timesteps(self, *args, **kwargs):
                super().set_timesteps(*args, **kwargs)
                self.sigmas = self.sigmas[::self.stride].clone()
                self.timesteps = (self.sigmas[:-1] * self.config.num_train_timesteps).to(self.sigmas.device)
                self.num_inference_steps = len(self.timesteps)

        # The pipeline inspects set_timesteps for a `sigmas` parameter: expose the parent's exact signature.
        import inspect as signature_tools
        SubsetScheduler.set_timesteps.__signature__ = signature_tools.signature(FlowMatchEulerDiscreteScheduler.set_timesteps)
        default_scheduler = pipe.scheduler
        student_scheduler = SubsetScheduler.from_config(default_scheduler.config)
        student_scheduler.stride = stride
        report['checkpoints'] = []
        report['losses'] = []
        started = time.perf_counter()

        def evaluate(step):
            # Quantitative: K-step final latent against the teacher's 40-step final latent (same noise), 45 held-out prompts.
            # Qualitative: the same 45 images (four-seed text grids and 25 general prompts).
            name = f'step_{step:04d}'
            pipe.scheduler = student_scheduler
            errors, teacher_errors = {}, {}
            for row in request['rows']:
                final = {}

                def on_step_end(pipeline, index, timestep, tensors):
                    if index == K - 1:
                        final['latents'] = tensors['latents'].detach().float().cpu()
                    return {}

                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], num_inference_steps=request['teacher_steps'],
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil', callback_on_step_end=on_step_end).images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - t0
                if (pipe.scheduler.sigmas.detach().float().cpu() - held_grid[row['source_id']]).abs().max() > 1e-6:
                    raise RuntimeError('Student sigma grid differs from the teacher subset.')
                reference = held_final[row['source_id']]
                error = ((final['latents'][0] - reference).norm() / reference.norm()).item()
                errors[row['source_id']] = error
                teacher_errors[row['source_id']] = ((final['latents'][0] - teacher_final[row['source_id']]).norm() / teacher_final[row['source_id']].norm()).item()
                path = directory / (row['source_id'] + '-' + name + '.png')
                image.save(path, format='PNG')
                outputs[name][row['source_id']] = {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size,
                                                   'seconds': seconds, 'endpoint_relative_l2': error}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            text_ids = {r['source_id'] for r in request['rows'] if r['subdimension'] in ('text_rendering', 'posters')}
            mean = lambda values: sum(values) / len(values)
            entry = dict(step=step, seconds=time.perf_counter() - started,
                         endpoint_mean=mean(list(errors.values())),
                         endpoint_text=mean([e for s, e in errors.items() if s in text_ids]),
                         endpoint_general=mean([e for s, e in errors.items() if s not in text_ids]),
                         teacher_mean=mean(list(teacher_errors.values())),
                         teacher_text=mean([e for s, e in teacher_errors.items() if s in text_ids]),
                         teacher_general=mean([e for s, e in teacher_errors.items() if s not in text_ids]))
            if step > 0:
                path = directory / f'lora_{K}steps_v1_step_{step:04d}.pt'
                temporary = path.with_suffix('.tmp')
                torch.save({'request_key': key, 'step': step, 'continued_from': request.get('init_lora'), 'student_steps': K, 'lora': lora, 'bridge_dtype': request['bridge_dtype'],
                            'state_dict': {n: p.detach().cpu().clone() for n, p in trainable.items()}}, temporary)
                temporary.replace(path)
                entry['checkpoint'] = dict(name=path.name, sha256=digest(path), bytes=path.stat().st_size)
                report['artifacts'].append(entry['checkpoint'])
            report['checkpoints'].append(entry)
            print(f"{K}-step LoRA (version 1 targets), step {step}: teacher {entry['teacher_mean']:.4f} (text {entry['teacher_text']:.4f}, general {entry['teacher_general']:.4f}); version 1 endpoint {entry['endpoint_mean']:.4f} (text {entry['endpoint_text']:.4f}, "
                  f"general {entry['endpoint_general']:.4f}) ({entry['seconds']:.0f}s)", flush=True)
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()

        # Smoke test of the training path before any evaluation: one forward and backward, gradients discarded, no update.
        with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
            train_loss(train_items[0], 0).backward()
        if not all(p.grad is not None for p in trainable.values()):
            raise RuntimeError('Some adapters received no gradient.')
        for parameter in trainable.values():
            parameter.grad = None
        optimizer = torch.optim.AdamW(list(trainable.values()), lr=fit['learning_rate'], weight_decay=0.0)
        rng = random.Random(fit['sampling_seed'])
        running = []
        for step in range(fit['updates'] + 1):
            if step % fit['checkpoint_every'] == 0:
                model.eval()
                evaluate(step)
            if step == fit['updates']:
                break
            for group in optimizer.param_groups:
                # Short warmup, then cosine decay to zero over the run.
                group['lr'] = fit['learning_rate'] * min(1.0, (step + 1) / fit['warmup']) * 0.5 * (1 + math.cos(math.pi * step / fit['updates']))
            optimizer.zero_grad(set_to_none=True)
            # Backward recomputes checkpointed blocks: keep the forward attention kernel and cache context.
            with model.cache_context('cond'), sdpa_kernel([SDPBackend.MATH]):
                for _ in range(fit['accumulation']):
                    loss = train_loss(rng.choice(train_items), rng.randrange(K)) / fit['accumulation']
                    loss.backward()
                    running.append(loss.item() * fit['accumulation'])
            torch.nn.utils.clip_grad_norm_(list(trainable.values()), fit['gradient_clip'])
            optimizer.step()
            report['optimizer_updates'] += 1
            if len(running) >= 50:
                report['losses'].append({'step': step + 1, 'mean_loss': sum(running) / len(running)})
                running = []
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_v1_lora():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    base_request = load(RUN_L8 + '/request.json')['request']
    v1 = load(RUN_V1 + '/summary.json')
    v1_request = load(RUN_V1 + '/request.json')['request']
    teacher = load(RUN + '/summary.json')
    ocr = load(RUN_OCR_V1 + '/summary.json')
    held = [r for r in v1_request['rows'] if r['split'] == 'held_out']
    request = {k: v for k, v in base_request.items() if k not in ('worker_source', 'version', 'fit', 'scope', 'selection', 'trajectories', 'rows')}
    request.update(
        version='phase10-lightning-v1-targets-v1', worker_source=SOURCES['lightning_v1_worker.py'],
        trajectories=dict(remote_directory=f"/phase3/phase10/v1_trajectories/{v1['request_key']}",
                          train_ids=[r['source_id'] for r in v1_request['rows'] if r['split'] == 'train'],
                          sha256={r['source_id']: r['trajectory']['sha256'] for r in v1['rows']}),
        teacher_reference=dict(remote_directory=f"/phase3/phase10/teacher_trajectories/{teacher['request_key']}",
                               sha256={r['source_id']: r['trajectory']['sha256'] for r in teacher['rows'] if r['split'] == 'held_out'}),
        rows=held,
        text_masks=dict(remote_path=f"/phase3/phase10/text_regions/{ocr['request_key']}/{ocr['masks']['name']}", sha256=ocr['masks']['sha256']),
        fit=dict(init_seed=20261019, sampling_seed=20261031, learning_rate=1e-4, warmup=20, gradient_clip=1.0,
                 updates=3000, checkpoint_every=500, accumulation=2, text_weight=10.0),
        selection='No automatic selection: every checkpoint saved; distance to version 1 and to the teacher on 45 held-out prompts and their images; spelling and structure decide',
        scope='Fresh 8-step LoRA trained on version 1 own 40-step trajectories (500 text, 500 general), text weighted 10x, cosine learning rate')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/lightning_v1' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/lightning_v1/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-lightning-v1')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'lightning_v1_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=43200, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'trainable_parameters')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN_LV1 = 'data/phase10/lightning_v1/' + main_v1_lora()
LAUNCH_ARGS = []
summary_lv1 = reuse_or_launch(RUN_LV1, main_v1_lora)

## 9. Drift test: are the structure failures made in the first steps?

Three failures (the star, the cyclist, the checkerboard) look the same at every checkpoint of step 8. If they come
from drift, the student's first jumps land slightly off and the later steps never recover. Test: on the five failure
prompts, the 8-step LoRA (update 2000) starts not from noise but from version 1's own recorded latent after 0, 5, 10 or
20 of its 40 steps, and does the remaining student steps. If the images come out right when the first jumps are taken
for it, the failures are made early and training on the student's own states (on-policy) is the fix; if they stay
broken, the later steps are at fault.

In [ ]:
# Drift test: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run steps 6 and 8 first).
DRIFT_UPDATE, DRIFT_ROWS = 2000, [1, 2, 10, 12, 18]   # checkerboard, star, four friends, cyclist, alarm clock
SOURCES['drift_test_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/drift_test') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'start{s:02d}' for s in request['starts']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # Version 1 (BF16 bridges) with the 8-step LoRA, sampled by hand: Euler steps on the student's grid (every 5th point of the
        # 40-step grid), starting from version 1's own recorded latent at grid index `start` (0 = from noise, as in normal use).
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=False)
            bridge_slots.append(bridge_spec['layer'])
        lora_spec = request['lora']
        lora_path = Path(lora_spec['remote_path'])
        if digest(lora_path) != lora_spec['sha256']:
            raise RuntimeError('LoRA checkpoint differs.')
        lora_state = torch.load(lora_path, weights_only=True, map_location='cpu')
        rank, alpha = lora_state['lora']['rank'], lora_state['lora']['alpha']
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), rank, alpha))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], rank, alpha)
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if sorted(trainable) != sorted(lora_state['state_dict']):
            raise RuntimeError('LoRA structure differs.')
        with torch.no_grad():
            for name, parameter in trainable.items():
                parameter.copy_(lora_state['state_dict'][name])
        model.requires_grad_(False)
        model.eval()
        gc.collect()
        torch.cuda.empty_cache()
        latents_mean = torch.tensor(pipe.vae.config.latents_mean).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        latents_std = torch.tensor(pipe.vae.config.latents_std).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        traj_dir = Path(request['trajectories']['remote_directory'])
        stride = request['teacher_steps'] // request['student_steps']
        report['distances'] = {}
        for row in request['rows']:
            path = traj_dir / (row['source_id'] + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][row['source_id']]:
                raise RuntimeError('Trajectory differs: ' + row['source_id'])
            item = torch.load(path, weights_only=True, map_location='cpu')
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            reference = item['latents'][-1].float()
            for start_index in request['starts']:
                name = f'start{start_index:02d}'
                x = item['latents'][start_index].unsqueeze(0).to('cuda').float()
                cache = QwenImage21KVCache(len(model.transformer_blocks))
                first = True
                with torch.inference_mode(), model.cache_context('cond'):
                    for index in range(start_index, request['teacher_steps'], stride):
                        sigma, sigma_next = item['sigmas'][index].item(), item['sigmas'][index + stride].item()
                        timestep = torch.tensor([item['timesteps'][index]], device='cuda', dtype=torch.bfloat16)
                        velocity = model(hidden_states=x.to(torch.bfloat16), timestep=timestep,
                                         encoder_hidden_states=cond['encoder_hidden_states'],
                                         encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                                         img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                                         kv_cache=cache, kv_cache_mode='extract' if first else 'cached',
                                         return_dict=False)[0][:, -x.shape[1]:].float()
                        first = False
                        x = (x + (sigma_next - sigma) * velocity).to(torch.bfloat16).float()
                    report['distances'].setdefault(row['source_id'], {})[name] = ((x[0].cpu() - reference).norm() / reference.norm()).item()
                    latent = pipe._unpack_latents(x.to(torch.bfloat16), row['height'], row['width'], pipe.vae_scale_factor).to(pipe.vae.dtype)
                    latent = latent * latents_std.to(latent.device, latent.dtype) + latents_mean.to(latent.device, latent.dtype)
                    decoded = pipe.vae.decode(latent, return_dict=False)[0][:, :, 0]
                    image = pipe.image_processor.postprocess(decoded, output_type='pil')[0].convert('RGB')
                out = directory / (row['source_id'] + '-' + name + '.png')
                image.save(out, format='PNG')
                outputs[name][row['source_id']] = {'name': out.name, 'sha256': digest(out), 'bytes': out.stat().st_size}
                report['calls'][name] += 1
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_drift():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    v1 = load(RUN_V1 + '/summary.json')
    v1_request = load(RUN_V1 + '/request.json')['request']
    lora_request = load(RUN_LV1 + '/request.json')['request']
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key_lv1 = RUN_LV1.rstrip('/').split('/')[-1]
    progress = json.loads(b''.join(volume.read_file(f'/phase10/lightning_v1/{key_lv1}/progress.json')))
    checkpoint = next(c['checkpoint'] for c in progress['checkpoints'] if c['step'] == DRIFT_UPDATE)
    first = json.loads(next((root / 'data/phase7/bridge_images').glob('*/summary.json')).read_bytes())
    failure_ids = [first['rows'][i]['source_id'] for i in DRIFT_ROWS]
    rows = [r for r in v1_request['rows'] if r['split'] == 'held_out' and r['source_id'] in failure_ids]
    request = {k: v1_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                          'canonical_tables_key', 'core_source', 'wrapper_source', 'teacher_steps',
                                          'conditioned_bridge_source', 'bridges')}
    request.update(
        version='phase10-drift-test-v1', worker_source=SOURCES['drift_test_worker.py'], execution_account='new', candidate_only=True,
        components_source=lora_request['components_source'], student_steps=8, starts=[0, 5, 10, 20], rows=rows,
        lora=dict(update=DRIFT_UPDATE, remote_path=f'/phase3/phase10/lightning_v1/{key_lv1}/{checkpoint["name"]}', sha256=checkpoint['sha256']),
        trajectories=dict(remote_directory=f"/phase3/phase10/v1_trajectories/{v1['request_key']}",
                          sha256={r['source_id']: r['trajectory']['sha256'] for r in v1['rows'] if r['source_id'] in failure_ids}),
        scope='Drift test: 8-step LoRA started from version 1 own latent after 0, 5, 10 or 20 of 40 steps, on the five structure-failure prompts')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/drift_test' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    set_modal_account('current', purpose='compute')
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/drift_test/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-drift-test')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'drift_test_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=7200, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'distances')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN_DRIFT = 'data/phase10/drift_test/' + main_drift()
LAUNCH_ARGS = []
summary_drift = reuse_or_launch(RUN_DRIFT, main_drift)

## 10. Hybrid sampling: version 1 takes the first jump

The drift test showed that the five structure failures are made entirely in the first jump, from noise to grid point
5: started from version 1's own latent at point 5, the LoRA draws all five right. So version 1 alone (LoRA off) takes
the start in k small Euler steps along its own 40-step grid, and the LoRA does the remaining 7 jumps:
k = 1 (points 0-5), 2 (0-2-5), 3 (0-1-3-5) or 5 (0-1-2-3-4-5), i.e. 8, 9, 10 or 12 model calls instead of 40. The LoRA
is switched off by setting its scale to 0; each mode keeps its own prompt cache. All 45 held-out prompts, distance to
version 1 at 40 steps and the images.

In [ ]:
# Hybrid sampling: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run steps 6, 8 and 9 first).
SOURCES['hybrid_test_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/hybrid_test') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'hybrid{int(k)}' for k in request['hybrid_paths']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # Version 1 (BF16 bridges) with the 8-step LoRA, sampled by hand: Euler steps on the student's grid (every 5th point of the
        # 40-step grid), starting from version 1's own recorded latent at grid index `start` (0 = from noise, as in normal use).
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=False)
            bridge_slots.append(bridge_spec['layer'])
        lora_spec = request['lora']
        lora_path = Path(lora_spec['remote_path'])
        if digest(lora_path) != lora_spec['sha256']:
            raise RuntimeError('LoRA checkpoint differs.')
        lora_state = torch.load(lora_path, weights_only=True, map_location='cpu')
        rank, alpha = lora_state['lora']['rank'], lora_state['lora']['alpha']
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), rank, alpha))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], rank, alpha)
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if sorted(trainable) != sorted(lora_state['state_dict']):
            raise RuntimeError('LoRA structure differs.')
        with torch.no_grad():
            for name, parameter in trainable.items():
                parameter.copy_(lora_state['state_dict'][name])
        lora_modules = [m for m in model.modules() if isinstance(m, components['MasterLoRALinear'])]
        lora_scale = lora_modules[0].scale
        if len(lora_modules) != 30 * 4:
            raise RuntimeError('LoRA module count differs.')
        model.requires_grad_(False)
        model.eval()
        gc.collect()
        torch.cuda.empty_cache()
        latents_mean = torch.tensor(pipe.vae.config.latents_mean).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        latents_std = torch.tensor(pipe.vae.config.latents_std).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        traj_dir = Path(request['trajectories']['remote_directory'])
        stride = request['teacher_steps'] // request['student_steps']
        report['distances'] = {}
        for row in request['rows']:
            path = traj_dir / (row['source_id'] + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][row['source_id']]:
                raise RuntimeError('Trajectory differs: ' + row['source_id'])
            item = torch.load(path, weights_only=True, map_location='cpu')
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            reference = item['latents'][-1].float()
            for k_name, path_points in request['hybrid_paths'].items():
                name = f'hybrid{int(k_name)}'
                # Version 1's own path points first (LoRA off), then the student's grid from point `handover` (LoRA on).
                points = list(path_points) + list(range(request['handover'] + stride, request['teacher_steps'] + 1, stride))
                x = item['latents'][0].unsqueeze(0).to('cuda').float()
                caches = {False: QwenImage21KVCache(len(model.transformer_blocks)), True: QwenImage21KVCache(len(model.transformer_blocks))}
                extracted = {False: False, True: False}
                with torch.inference_mode(), model.cache_context('cond'):
                    for a, b in zip(points[:-1], points[1:]):
                        lora_on = a >= request['handover']
                        for module in lora_modules:
                            module.scale = lora_scale if lora_on else 0.0
                        sigma, sigma_next = item['sigmas'][a].item(), item['sigmas'][b].item()
                        timestep = torch.tensor([item['timesteps'][a]], device='cuda', dtype=torch.bfloat16)
                        velocity = model(hidden_states=x.to(torch.bfloat16), timestep=timestep,
                                         encoder_hidden_states=cond['encoder_hidden_states'],
                                         encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                                         img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                                         kv_cache=caches[lora_on], kv_cache_mode='cached' if extracted[lora_on] else 'extract',
                                         return_dict=False)[0][:, -x.shape[1]:].float()
                        extracted[lora_on] = True
                        x = (x + (sigma_next - sigma) * velocity).to(torch.bfloat16).float()
                    report.setdefault('model_calls', {})[name] = len(points) - 1
                    report['distances'].setdefault(row['source_id'], {})[name] = ((x[0].cpu() - reference).norm() / reference.norm()).item()
                    latent = pipe._unpack_latents(x.to(torch.bfloat16), row['height'], row['width'], pipe.vae_scale_factor).to(pipe.vae.dtype)
                    latent = latent * latents_std.to(latent.device, latent.dtype) + latents_mean.to(latent.device, latent.dtype)
                    decoded = pipe.vae.decode(latent, return_dict=False)[0][:, :, 0]
                    image = pipe.image_processor.postprocess(decoded, output_type='pil')[0].convert('RGB')
                out = directory / (row['source_id'] + '-' + name + '.png')
                image.save(out, format='PNG')
                outputs[name][row['source_id']] = {'name': out.name, 'sha256': digest(out), 'bytes': out.stat().st_size}
                report['calls'][name] += 1
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_hybrid():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    drift = load(RUN_DRIFT + '/request.json')['request']
    v1 = load(RUN_V1 + '/summary.json')
    v1_request = load(RUN_V1 + '/request.json')['request']
    rows = [r for r in v1_request['rows'] if r['split'] == 'held_out']
    request = {k: v for k, v in drift.items() if k not in ('worker_source', 'version', 'starts', 'rows', 'trajectories', 'scope')}
    request.update(
        version='phase10-hybrid-test-v1', worker_source=SOURCES['hybrid_test_worker.py'], rows=rows, handover=5,
        hybrid_paths={'1': [0, 5], '2': [0, 2, 5], '3': [0, 1, 3, 5], '5': [0, 1, 2, 3, 4, 5]},
        trajectories=dict(remote_directory=f"/phase3/phase10/v1_trajectories/{v1['request_key']}",
                          sha256={r['source_id']: r['trajectory']['sha256'] for r in v1['rows'] if r['split'] == 'held_out'}),
        scope='Hybrid sampling: version 1 alone to grid point 5 of 40 in 1, 2, 3 or 5 steps, then the 8-step LoRA; 45 held-out prompts')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/hybrid_test' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/hybrid_test/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-hybrid-test')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'hybrid_test_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=7200, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'model_calls')}), flush=True)
    return request_key


def hybrid_table(report):
    import statistics
    names = list(report['model_calls'])
    text = [s for s in report['distances'] if any(t in s for t in ('2fa54a3c4eb8', 'c9344c076abf', 'e83d7dcacb2d', 'f66dd8002f66', '0b32a45e0b90'))]
    head = '| Sampler | Model calls | Median distance, all 45 | Text (20) | General (25) |\n|---|---|---|---|---|\n'
    lines = []
    for n in names:
        med = lambda ids: statistics.median(report['distances'][s][n] for s in ids)
        general = [s for s in report['distances'] if s not in text]
        lines.append(f"| {n} | {report['model_calls'][n]} | {100 * med(list(report['distances'])):.1f}% | {100 * med(text):.1f}% | {100 * med(general):.1f}% |")
    display(Markdown(head + chr(10).join(lines)))

LAUNCH_ARGS = ['--dry-run']
RUN_HYBRID = 'data/phase10/hybrid_test/' + main_hybrid()
LAUNCH_ARGS = []
summary_hybrid = reuse_or_launch(RUN_HYBRID, main_hybrid)
if summary_hybrid and summary_hybrid.get('status') == 'passed':
    hybrid_table(summary_hybrid)

## 10b. Can a fused model do it?

Fusing the LoRA into the weights leaves no LoRA-free model for the start-up. Here the start-up steps of step 10 run with
the LoRA on as well, which is what one fused model with an uneven schedule (small steps first) would do. If it matches
step 10, version 1 ships fused with that schedule; if not, the LoRA stays separate (switched off for the start-up) or is
retrained on the uneven schedule.

In [ ]:
# Fused hybrid: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run step 10 first).
SOURCES['hybrid_fused_test_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/hybrid_fused_test') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f'hybrid{int(k)}' for k in request['hybrid_paths']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # Version 1 (BF16 bridges) with the 8-step LoRA, sampled by hand: Euler steps on the student's grid (every 5th point of the
        # 40-step grid), starting from version 1's own recorded latent at grid index `start` (0 = from noise, as in normal use).
        from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVLayerCache
        outputs = {v: {} for v in variants}
        conditioned, components = {}, {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        exec(compile(request['components_source'], 'healing_components.py', 'exec'), components)
        bridge_slots = []
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=False)
            bridge_slots.append(bridge_spec['layer'])
        lora_spec = request['lora']
        lora_path = Path(lora_spec['remote_path'])
        if digest(lora_path) != lora_spec['sha256']:
            raise RuntimeError('LoRA checkpoint differs.')
        lora_state = torch.load(lora_path, weights_only=True, map_location='cpu')
        rank, alpha = lora_state['lora']['rank'], lora_state['lora']['alpha']
        for slot, block in enumerate(list(model.transformer_blocks)):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                setattr(block.attn, name, components['MasterLoRALinear'](getattr(block.attn, name), rank, alpha))
            block.attn.to_out[0] = components['MasterLoRALinear'](block.attn.to_out[0], rank, alpha)
            model.transformer_blocks[slot] = components['FunctionalCheckpointBlock'](block, QwenImage21KVLayerCache)
        trainable = {n: p for n, p in model.named_parameters() if '.attn.' in n and ('.down.' in n or '.up.' in n)}
        if sorted(trainable) != sorted(lora_state['state_dict']):
            raise RuntimeError('LoRA structure differs.')
        with torch.no_grad():
            for name, parameter in trainable.items():
                parameter.copy_(lora_state['state_dict'][name])
        lora_modules = [m for m in model.modules() if isinstance(m, components['MasterLoRALinear'])]
        lora_scale = lora_modules[0].scale
        if len(lora_modules) != 30 * 4:
            raise RuntimeError('LoRA module count differs.')
        model.requires_grad_(False)
        model.eval()
        gc.collect()
        torch.cuda.empty_cache()
        latents_mean = torch.tensor(pipe.vae.config.latents_mean).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        latents_std = torch.tensor(pipe.vae.config.latents_std).view(1, pipe.vae.config.z_dim, 1, 1, 1)
        traj_dir = Path(request['trajectories']['remote_directory'])
        stride = request['teacher_steps'] // request['student_steps']
        report['distances'] = {}
        for row in request['rows']:
            path = traj_dir / (row['source_id'] + '-trajectory.pt')
            if digest(path) != request['trajectories']['sha256'][row['source_id']]:
                raise RuntimeError('Trajectory differs: ' + row['source_id'])
            item = torch.load(path, weights_only=True, map_location='cpu')
            cond = {k: v.to('cuda') if torch.is_tensor(v) else v for k, v in item['conditioning'].items()}
            reference = item['latents'][-1].float()
            for k_name, path_points in request['hybrid_paths'].items():
                name = f'hybrid{int(k_name)}'
                # Version 1's own path points first (LoRA off), then the student's grid from point `handover` (LoRA on).
                points = list(path_points) + list(range(request['handover'] + stride, request['teacher_steps'] + 1, stride))
                x = item['latents'][0].unsqueeze(0).to('cuda').float()
                caches = {False: QwenImage21KVCache(len(model.transformer_blocks)), True: QwenImage21KVCache(len(model.transformer_blocks))}
                extracted = {False: False, True: False}
                with torch.inference_mode(), model.cache_context('cond'):
                    for a, b in zip(points[:-1], points[1:]):
                        lora_on = request['lora_in_startup'] or a >= request['handover']
                        for module in lora_modules:
                            module.scale = lora_scale if lora_on else 0.0
                        sigma, sigma_next = item['sigmas'][a].item(), item['sigmas'][b].item()
                        timestep = torch.tensor([item['timesteps'][a]], device='cuda', dtype=torch.bfloat16)
                        velocity = model(hidden_states=x.to(torch.bfloat16), timestep=timestep,
                                         encoder_hidden_states=cond['encoder_hidden_states'],
                                         encoder_hidden_states_mask=cond['encoder_hidden_states_mask'],
                                         img_shapes=cond['img_shapes'], img_mask=cond['img_mask'], attention_kwargs=None,
                                         kv_cache=caches[lora_on], kv_cache_mode='cached' if extracted[lora_on] else 'extract',
                                         return_dict=False)[0][:, -x.shape[1]:].float()
                        extracted[lora_on] = True
                        x = (x + (sigma_next - sigma) * velocity).to(torch.bfloat16).float()
                    report.setdefault('model_calls', {})[name] = len(points) - 1
                    report['distances'].setdefault(row['source_id'], {})[name] = ((x[0].cpu() - reference).norm() / reference.norm()).item()
                    latent = pipe._unpack_latents(x.to(torch.bfloat16), row['height'], row['width'], pipe.vae_scale_factor).to(pipe.vae.dtype)
                    latent = latent * latents_std.to(latent.device, latent.dtype) + latents_mean.to(latent.device, latent.dtype)
                    decoded = pipe.vae.decode(latent, return_dict=False)[0][:, :, 0]
                    image = pipe.image_processor.postprocess(decoded, output_type='pil')[0].convert('RGB')
                out = directory / (row['source_id'] + '-' + name + '.png')
                image.save(out, format='PNG')
                outputs[name][row['source_id']] = {'name': out.name, 'sha256': digest(out), 'bytes': out.stat().st_size}
                report['calls'][name] += 1
        results_volume.commit()
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_fused():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    base = json.loads((root / RUN_HYBRID / 'request.json').read_text(encoding='utf-8'))['request']
    request = {k: v for k, v in base.items() if k not in ('worker_source', 'version', 'scope')}
    request.update(version='phase10-hybrid-fused-test-v1', worker_source=SOURCES['hybrid_fused_test_worker.py'], lora_in_startup=True,
                   scope='As step 10 with the LoRA on in the start-up steps too: what a single fused model with an uneven schedule does')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/hybrid_fused_test' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='compute')
    import modal
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/hybrid_fused_test/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-hybrid-fused-test')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'hybrid_fused_test_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=7200, cpu=8, memory=98304,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls', 'model_calls')}), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN_FUSED = 'data/phase10/hybrid_fused_test/' + main_fused()
LAUNCH_ARGS = []
summary_fused = reuse_or_launch(RUN_FUSED, main_fused)
if summary_fused and summary_fused.get('status') == 'passed':
    hybrid_table(summary_fused)

## 11. The shipping path: fused LoRA, stock pipeline, 10 steps

Version 1 ships with the LoRA fused into the weights (W + B·A, 120 attention layers) and a 10-step schedule: points
0, 1, 3, 5, 10, 15, 20, 25, 30 and 35 of the stock 40-step schedule for the image size, then the end. Two ways to run it:

* **A. A small scheduler class** that builds the 40-step schedule and keeps those points (the reference implementation).
* **B. Plain sigmas** from the helper `nihonga_sigmas(config, width, height)` below, passed as `pipe(..., sigmas=...)` to
  the stock pipeline with a neutral scheduler (no shift, no terminal stretch), so nothing is applied twice; in ComfyUI
  the same list goes into a custom-sigmas node.

This step runs B exactly as a user would: fused weights, stock `QwenImage21Pipeline`, sigmas from the helper. It checks
that the scheduler used exactly the helper's values, that update 2000 reproduces step 10b, and evaluates update 2500.

In [ ]:
# Shipping path: reuses the saved run; GPU only when the result is missing and ALLOW_GPU is True (run steps 6 and 8 first).
SOURCES['nihonga_sigmas.py'] = r'''def nihonga_sigmas(scheduler_config, width, height, points=(0, 1, 3, 5, 10, 15, 20, 25, 30, 35), base_steps=40):
    """Version 1's 10-step schedule: points of the stock 40-step schedule for this image size (route B).
    Use with a neutral scheduler: FlowMatchEulerDiscreteScheduler.from_config(config, use_dynamic_shifting=False,
    shift=1.0, shift_terminal=None), then pipe(..., sigmas=nihonga_sigmas(config, width, height))."""
    import numpy as np
    from diffusers import FlowMatchEulerDiscreteScheduler
    from diffusers.pipelines.qwenimage21.pipeline_qwenimage21 import calculate_shift
    reference = FlowMatchEulerDiscreteScheduler.from_config(scheduler_config)
    mu = calculate_shift((height // 16) * (width // 16), reference.config.get('base_image_seq_len', 256),
                         reference.config.get('max_image_seq_len', 4096), reference.config.get('base_shift', 0.5),
                         reference.config.get('max_shift', 1.15))
    reference.set_timesteps(sigmas=np.linspace(1.0, 1 / base_steps, base_steps), mu=mu)
    return [float(reference.sigmas[i]) for i in points]
'''
exec(SOURCES["nihonga_sigmas.py"])
SHIP_UPDATES = [2000, 2500]
SOURCES['shipping_test_worker.py'] = r'''def generate_multiseed_text(request, key):
    """Extra seeds for the text prompts: teacher, B, and C with the best block-13 bridge.
    Same prompt, CPU noise seed, size, scheduler and 40 steps for every model. Qualitative screen only."""
    import gc
    import hashlib
    import inspect
    import json
    import os
    import time
    from pathlib import Path

    import torch
    from diffusers import QwenImage21Pipeline
    from diffusers.models.transformers.transformer_qwenimage21 import QwenImage21KVCache, QwenImage21KVLayerCache

    directory = Path('/phase3/phase10/shipping_test') / key
    directory.mkdir(parents=True, exist_ok=True)
    report_path = directory / 'summary.json'
    if report_path.exists():
        return json.loads(report_path.read_text())
    if (directory / 'started.json').exists():
        raise RuntimeError('Incomplete image attempt exists; inspect before repeating calls.')
    (directory / 'started.json').write_text(json.dumps({'request_key': key}))
    (directory / 'request.json').write_text(json.dumps({'request_key': key, 'request': request}, indent=2))
    results_volume.commit()

    def digest(path):
        h = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                h.update(chunk)
        return h.hexdigest()

    def table_hash(value):
        return hashlib.sha256(memoryview(value.detach().cpu().contiguous().view(torch.uint8).numpy())).hexdigest()

    variants = [f"fused_u{l['update']}" for l in request['loras']]
    report = {'request_key': key, 'status': 'running', 'rows': [], 'artifacts': [],
              'calls': {v: 0 for v in variants}, 'optimizer_updates': 0}
    try:
        if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
            raise RuntimeError('Deterministic workspace environment differs.')
        teacher = request['teacher']
        report['gpu'] = torch.cuda.get_device_name(0)
        if 'H100' not in report['gpu']:
            raise RuntimeError('Expected H100.')
        pipe = QwenImage21Pipeline.from_pretrained(teacher['model_repo'], revision=teacher['model_revision'],
                                                   dtype=torch.bfloat16, local_files_only=True).to('cuda')
        pipe.set_progress_bar_config(disable=True)
        model = pipe.transformer.eval()
        for component in (model, pipe.text_encoder, pipe.vae):
            component.requires_grad_(False)
        torch.use_deterministic_algorithms(True)
        torch.set_float32_matmul_precision('highest')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
        torch.backends.cuda.allow_fp16_bf16_reduction_math_sdp(False)
        if digest(Path(inspect.getfile(type(model)))) != request['transformer_source_sha256']:
            raise RuntimeError('Transformer source differs.')
        config_path = (Path('/root/.cache/huggingface/hub') / ('models--' + teacher['model_repo'].replace('/', '--')) /
                       'snapshots' / teacher['model_revision'] / 'transformer/config.json')
        if digest(config_path) != teacher['config_sha256'] or len(model.transformer_blocks) != 32:
            raise RuntimeError('Teacher configuration differs.')
        # Derived position/time tables: use the audited values on every host (as in steps 7-11).
        derived = {f'pos_embed.freqs.{axis}': value for axis, value in enumerate(model.pos_embed.freqs)}
        derived['time_text_embed.time_proj.freqs'] = model.time_text_embed.time_proj.freqs
        expected = request['expected_derived_tables']
        mismatched = sorted(n for n, v in derived.items() if table_hash(v) != expected[n])
        if mismatched:
            canonical = torch.load(Path('/phase3/phase7/canonical_derived_tables') / (request['canonical_tables_key'] + '.pt'),
                                   weights_only=True, map_location='cpu')
            if any(table_hash(canonical[n]) != expected[n] for n in expected):
                raise RuntimeError('Canonical derived tables differ from the audited hashes.')
            original = model.pos_embed.freqs
            replaced = [canonical[f'pos_embed.freqs.{a}'].to(device=v.device, dtype=v.dtype) for a, v in enumerate(original)]
            model.pos_embed.freqs = tuple(replaced) if isinstance(original, tuple) else replaced
            with torch.no_grad():
                model.time_text_embed.time_proj.freqs.copy_(canonical['time_text_embed.time_proj.freqs'])
            if any(table_hash(v) != expected[n] for n, v in
                   list({f'pos_embed.freqs.{a}': v for a, v in enumerate(model.pos_embed.freqs)}.items()) +
                   [('time_text_embed.time_proj.freqs', model.time_text_embed.time_proj.freqs)]):
                raise RuntimeError('Derived tables still differ after loading the audited copy.')
            report['canonical_tables_applied'] = mismatched

        def generate(row, variant):
            path = directory / (row['source_id'] + '-' + variant + '.png')
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'],
                             num_inference_steps=40, true_cfg_scale=teacher['true_cfg_scale'],
                             generator=torch.Generator('cpu').manual_seed(row['seed']),
                             use_kv_cache=True, output_type='pil').images[0]
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            temporary = path.with_suffix('.tmp')
            image.save(temporary, format='PNG')
            temporary.replace(path)
            results_volume.commit()
            report['calls'][variant] += 1
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
            return {'name': path.name, 'sha256': digest(path), 'bytes': path.stat().st_size, 'seconds': elapsed}

        # ---- Shipping path: version 1 (BF16 bridges), LoRA fused into the weights, stock pipeline, sigmas from the helper. ----
        from diffusers import FlowMatchEulerDiscreteScheduler
        outputs = {v: {} for v in variants}
        helper = {}
        exec(compile(request['sigmas_helper_source'], 'nihonga_sigmas.py', 'exec'), helper)
        conditioned = {}
        exec(compile(request['conditioned_bridge_source'], 'conditioned_bridge.py', 'exec'), conditioned)
        for bridge_spec in request['bridges']:
            path = Path(bridge_spec['remote_path'])
            if digest(path) != bridge_spec['sha256']:
                raise RuntimeError('Bridge checkpoint differs: ' + bridge_spec['remote_path'])
            checkpoint = torch.load(path, weights_only=True, map_location='cpu')
            if checkpoint['layer'] != bridge_spec['layer'] or checkpoint['step'] != bridge_spec['step']:
                raise RuntimeError('Bridge provenance differs.')
            variant = checkpoint['variant']
            bridge = conditioned['AttentionConditionedBridge'](4096, variant['rank'], variant['heads'], 128)
            bridge.load_state_dict(checkpoint['state_dict'])
            bridge.requires_grad_(False)
            conditioned['insert_conditioned_bridge'](model, bridge_spec['layer'], bridge.float(), keep_dtype=False)
        bridge_slots = {spec['layer'] for spec in request['bridges']}
        layers = {}
        for slot, block in enumerate(model.transformer_blocks):
            if slot in bridge_slots:
                continue
            for name in ('to_q', 'to_k', 'to_v'):
                layers[f'transformer_blocks.{slot}.inner.attn.{name}'] = getattr(block.attn, name)
            layers[f'transformer_blocks.{slot}.inner.attn.to_out.0'] = block.attn.to_out[0]
        if len(layers) != 120:
            raise RuntimeError('Layer count differs.')
        originals = {k: m.weight.detach().to('cpu').clone() for k, m in layers.items()}
        default_scheduler = pipe.scheduler
        neutral = FlowMatchEulerDiscreteScheduler.from_config(default_scheduler.config, use_dynamic_shifting=False, shift=1.0, shift_terminal=None)
        report['schedule_check'] = {}
        for lora_spec in request['loras']:
            path = Path(lora_spec['remote_path'])
            if digest(path) != lora_spec['sha256']:
                raise RuntimeError('LoRA checkpoint differs.')
            state = torch.load(path, weights_only=True, map_location='cpu')
            scale = state['lora']['alpha'] / state['lora']['rank']
            with torch.no_grad():
                for key, module in layers.items():
                    down = state['state_dict'][key + '.down.weight'].to('cuda', torch.float32)
                    up = state['state_dict'][key + '.up.weight'].to('cuda', torch.float32)
                    base = originals[key].to('cuda', torch.float32)
                    module.weight.copy_((base + scale * (up @ down)).to(module.weight.dtype))
            model.eval()
            name = f"fused_u{lora_spec['update']}"
            pipe.scheduler = neutral
            for row in request['rows']:
                sigmas = helper['nihonga_sigmas'](default_scheduler.config, row['width'], row['height'])
                torch.cuda.synchronize()
                started = time.perf_counter()
                with torch.inference_mode():
                    image = pipe(prompt=row['prompt'], width=row['width'], height=row['height'], sigmas=sigmas,
                                 true_cfg_scale=teacher['true_cfg_scale'], generator=torch.Generator('cpu').manual_seed(row['seed']),
                                 use_kv_cache=True, output_type='pil').images[0]
                torch.cuda.synchronize()
                seconds = time.perf_counter() - started
                used = pipe.scheduler.sigmas.detach().float().cpu()
                expected = torch.tensor(sigmas + [0.0])
                report['schedule_check'][f"{name}/{row['source_id']}"] = float((used - expected).abs().max())
                out = directory / (row['source_id'] + '-' + name + '.png')
                image.save(out, format='PNG')
                outputs[name][row['source_id']] = {'name': out.name, 'sha256': digest(out), 'bytes': out.stat().st_size, 'seconds': seconds}
                report['calls'][name] += 1
            pipe.scheduler = default_scheduler
            (directory / 'progress.json').write_text(json.dumps(report, indent=2))
            results_volume.commit()
        if max(report['schedule_check'].values()) > 1e-6:
            raise RuntimeError('The stock scheduler did not reproduce the helper sigmas.')
        rendered = [v for v in variants if outputs[v]]
        for row in request['rows']:
            report['rows'].append({**row, **{v: outputs[v][row['source_id']] for v in rendered}})
            report['artifacts'].extend(outputs[v][row['source_id']] for v in rendered)
        report['status'] = 'passed'
    except Exception as exc:
        report['status'] = 'failed'
        report['error'] = type(exc).__name__ + ': ' + str(exc)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=True) + '\n')
    results_volume.commit()
    return report
'''

def main_ship():
    import argparse
    import hashlib
    parser = argparse.ArgumentParser()
    parser.add_argument('--dry-run', action='store_true')
    args = parser.parse_args(LAUNCH_ARGS)
    root = ROOT
    load = lambda path: json.loads((root / path).read_text(encoding='utf-8'))
    v1_request = load(RUN_V1 + '/request.json')['request']
    from modal_accounts import set_modal_account
    set_modal_account('current', purpose='download')
    import modal
    volume = modal.Volume.from_name('nihonga-phase3-artifacts')
    key_lv1 = RUN_LV1.rstrip('/').split('/')[-1]
    progress = json.loads(b''.join(volume.read_file(f'/phase10/lightning_v1/{key_lv1}/progress.json')))
    checkpoints = {c['step']: c['checkpoint'] for c in progress['checkpoints'] if c['step'] > 0}
    request = {k: v1_request[k] for k in ('teacher', 'transformer_source_sha256', 'runtime_environment', 'expected_derived_tables',
                                          'canonical_tables_key', 'core_source', 'wrapper_source', 'conditioned_bridge_source', 'bridges')}
    request.update(
        version='phase10-shipping-test-v1', worker_source=SOURCES['shipping_test_worker.py'], execution_account='new', candidate_only=True,
        sigmas_helper_source=SOURCES['nihonga_sigmas.py'],
        loras=[dict(update=u, remote_path=f"/phase3/phase10/lightning_v1/{key_lv1}/{checkpoints[u]['name']}", sha256=checkpoints[u]['sha256'])
               for u in SHIP_UPDATES],
        rows=[r for r in v1_request['rows'] if r['split'] == 'held_out'],
        scope='Shipping path: LoRA fused into the weights, stock pipeline, 10-step schedule as plain sigmas from the helper')
    from experiment_runtime import pin_reference_runtime
    pin_reference_runtime(request, root / 'data/recovered/modal')
    request_key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    local = root / 'data/phase10/shipping_test' / request_key
    local.mkdir(parents=True, exist_ok=True)
    (local / 'request.json').write_text(json.dumps(dict(request_key=request_key, request=request), indent=2) + chr(10), encoding='utf-8')
    print('Request:', request_key, flush=True)
    if args.dry_run:
        return request_key
    set_modal_account('current', purpose='compute')
    runtime = request['teacher']['runtime']
    results_volume = modal.Volume.from_name(runtime['results_volume'])
    remote = '/phase10/shipping_test/' + request_key
    try:
        report = json.loads(b''.join(results_volume.read_file(remote + '/summary.json')))
        print('Reusing completed remote report; no GPU scheduled.', flush=True)
    except FileNotFoundError:
        app = modal.App('nihonga-phase10-shipping-test')
        image = (modal.Image.debian_slim(python_version=runtime['python_version']).apt_install('git')
                 .pip_install(*request['execution_packages']).env(request['runtime_environment']))
        weights = modal.Volume.from_name(runtime['weights_volume'])
        namespace = {'results_volume': results_volume}
        exec(compile(request['worker_source'], 'shipping_test_worker.py', 'exec'), namespace)
        worker = app.function(image=image, gpu='H100!', serialized=True, timeout=7200, cpu=8, memory=131072,
                              max_containers=1, retries=0,
                              volumes={'/phase3': results_volume,
                                       '/root/.cache/huggingface': weights.with_mount_options(read_only=True)})(namespace['generate_multiseed_text'])
        with modal.enable_output(), app.run(detach=True):
            call = worker.spawn(request, request_key)
            (local / 'call.json').write_text(json.dumps(dict(request_key=request_key, function_call_id=call.object_id, account='new'), indent=2) + chr(10))
            print('Call:', call.object_id, flush=True)
            report = call.get()
    (local / 'summary.json').write_text(json.dumps(report, indent=2) + chr(10), encoding='utf-8')
    print(json.dumps({k: report.get(k) for k in ('status', 'error', 'calls')}), flush=True)
    print('max schedule difference', max(report.get('schedule_check', {0: 0}).values()), flush=True)
    return request_key

LAUNCH_ARGS = ['--dry-run']
RUN_SHIP = 'data/phase10/shipping_test/' + main_ship()
LAUNCH_ARGS = []
summary_ship = reuse_or_launch(RUN_SHIP, main_ship)